## Piece of my Heart

In [ ]:
#| default_exp game/piece

In [ ]:
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
#from fasthtml.jupyter import get_host
from fastlite import *
import fasthtml.components as fc
import httpx
import random
import pandas as pd
import threading
import numpy as np
from dataclasses import dataclass

In [ ]:
from monsterui.all import *

In [ ]:
#| export
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover, User
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.core import Terrain, DrainageBasins
from HexMagic.styles import StyleCSS, SVGBuilder
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex, HexWrapper
from HexMagic.styles import StyleCSS,  SVGBuilder

from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord ,HexDragMap, HexTouchMap, HexRegion



In [ ]:
from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging
from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag, _map_point
from HexMagic.game.data import Piece, PieceType, Instruction

In [ ]:
app, rt, hexGameServer  = appRoutes()

## Helpers

In [ ]:
read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

In [ ]:
!cat ../../HexMagic/game/data.py

In [ ]:
!cat ../../docs/ll*

We now have
```python
@patch(as_prop=True)
def c2f(self: ZoomResult) -> dict:
    """Lazily compute and cache the coarse→fine index mapping."""
    if not hasattr(self, '_c2f') or self._c2f is None:
        self._c2f = self.invert_mapper()
    return self._c2f

@patch
def project_region(self: ZoomResult, region: HexRegion) -> HexRegion:
    """Project a coarse region onto the zoomed fine grid."""
    fine_hexes = set()
    for coarse_idx in region.hexes:
        if coarse_idx in self.c2f:
            fine_hexes.update(self.c2f[coarse_idx])
    return HexRegion(hexes=fine_hexes, hexGrid=self.terrain.hexGrid)

@patch
def map_point(self: ZoomResult, coarse_idx: int) -> int:
    """Map a single coarse index to a representative fine index (middle of mapped set)."""
    fs = self.c2f.get(coarse_idx)
    if not fs:
        return -1
    return fs[len(fs) // 2]
```

In [ ]:
@rt
def hello(name: str): return P(f"Hello {name}!")

webMe(Div(
    H3("Say Hello"),
    Form(
        Input(placeholder="Your name...", id='name'),
        Button("Send"),
        hx_get=hello, hx_target="#result"
    ),
    Div(id='result'),
))

#| export
@patch
def create_game(self: GameStorage, user_id, template_name="bayArea",
                kingdoms=5, lakes=1, radius=10) -> ActiveGame:
    logging.info(f"create_game: user={user_id} template={template_name}")
    tt = TerrainTemplate()
    terrain = getattr(tt, template_name)()
    
    terrain.carve_to_ocean(num_lakes=lakes)
    terrain.hexGrid.adjustRadius(radius)

    # GameBoard now creates cover + basins internally
    board = GameBoard(terrain, top_n=kingdoms)
    board.expand_kingdoms(max_rounds=50)
    
    # Save using the cover that GameBoard created
    board.cover.db = self
    board.cover.save(name=template_name)
    board.save(self, world_id=board.cover.ident)

    self.db.execute("UPDATE user SET activeWorld = ? WHERE id = ?",
                    [board.cover.ident, user_id])
    
    return ActiveGame(board=board, cover=board.cover, world_id=board.cover.ident)


_game_cache = {}
_cache_lock = threading.Lock()

@patch
def active_board(self: GameStorage, user_id) -> ActiveGame:
    with _cache_lock:
        if user_id in _game_cache:
            logging.info(f"active_board: cache hit for user {user_id}")
            return _game_cache[user_id]
    
    logging.info(f"active_board: cache miss, building for user {user_id}")
    row = self.db.execute("SELECT activeWorld FROM user WHERE id = ?", [user_id]).fetchone()
    if not row or not row[0]:
        return None
    
    world_id = row[0]
    try:
        # gameboard() now loads cover internally and reuses cover.basin
        board = self.gameboard(world_id)
        active = ActiveGame(board=board, cover=board.cover, world_id=world_id)
        
        with _cache_lock:
            _game_cache[user_id] = active
        
        return active
    except Exception as e:
        logging.error(f"active_board: FAILED: {e}", exc_info=True)
        return None


## Common Routes

@rt
def create_world(session, template_name: str, kingdoms: int = 5,
                 lakes: int = 1, radius: int = 10):
    uid = ensure_user(session)
    globalStore.create_game(uid, template_name=template_name,
                            kingdoms=kingdoms, lakes=lakes, radius=radius)
    logging.info(f"create world redirecting")
    return RedirectResponse('/game', status_code=303)


@rt
def create_world(session, template_name: str, kingdoms: int = 5,
                 lakes: int = 1, radius: int = 10):
    uid = ensure_user(session)
    logging.info(f"create_world: session keys={list(session.keys())}")
    invalidate_cache(uid)  # <-- clear old game
    globalStore.create_game(uid, template_name=template_name,
                            kingdoms=kingdoms, lakes=lakes, radius=radius)
    return RedirectResponse('/game', status_code=303)

In [ ]:
# Empty before we go

In [ ]:
@rt("/piece_detail/{id}")
def piece_detail(session, id: str):
    """Placeholder for piece detail — right panel."""
    return Div(
        H4(f"🧑 Piece", cls="font-bold"),
        P(f"ID: {id[:12]}…", cls=TextPresets.muted_sm),
        Divider(),
        P("Actions and stats coming soon", cls="opacity-50 text-sm"),
        cls="space-y-2"
    )


## `zoom_region_fast` — Deep Dive

This is the **main pipeline** for the question: *"I have a HexRegion on the coarse grid — give me a high-resolution Terrain covering just that region, with weather and watersheds."*

It orchestrates every piece we've built so far into a 10-step pipeline that stays in numpy-land as long as possible, only constructing the expensive `HexGrid` geometry once at the very end.

### The pipeline at a glance

```
HexRegion (coarse)
    │
    ▼
┌─────────────────┐
│ 1. Bounding box  │  O(|region|)
│ 2. Chunk refs    │  O(bbox area / spacing²)
│ 3. Stitch chunks │  O(chunks × chunk_size)  ← numpy block copies
│ 4. Build mapper  │  O(1) closure creation
│ 5. Region filter │  O(merged_size)  ← identifies fine hexes in region
│ 6. Crop proxy    │  O(cropped_size) ← numpy 2D slice
│ 7. Wrap mapper   │  O(1) closure wrapping
│ 8. Build Terrain │  O(cropped_size) ← HexGrid constructed ONCE here
│ 9. Project weather│  O(valid_fine_hexes) ← vectorized numpy scatter
│10. Project sheds │  O(valid_fine_hexes) ← dict grouping + lookup
└─────────────────┘
    │
    ▼
ZoomResult(terrain, basins, mapper, chunks_loaded)
```

### Step 1 — Bounding box with padding

Given a `HexRegion` (a set of coarse hex indices), we find the axis-aligned bounding box in row/col space:

$$r_{\min}, r_{\max}, c_{\min}, c_{\max} = \text{region\_bounding\_box}(\text{region})$$

Then we **pad** by `halo_rings + 1` in every direction (clamped to grid bounds). The padding ensures that when we zoom, fine hexes at the region boundary have enough neighbor data for interpolation and drainage flow:

$$r_{\min}' = \max(0,\; r_{\min} - p) \qquad r_{\max}' = \min(N_r - 1,\; r_{\max} + p)$$

where $p = \text{halo\_rings} + 1$.

**Why halo_rings?** Each chunk already has a halo of extra hexes for seamless stitching. The `+1` ensures we don't get edge artifacts on the region boundary itself.

### Step 2 — Chunk discovery

`bbox_to_chunk_refs` samples the padded bounding box at `spacing = 2 × rings` intervals and maps each sample point to its chunk via `world_to_chunk`:

```
Padded bbox on coarse grid:
┌─────────────────────────┐
│  ·  ·  ·  ·  ·  ·  ·   │   · = sample points at spacing intervals
│                         │
│  ·  ·  ·  ·  ·  ·  ·   │   Each · → world_to_chunk → ChunkRef
│                         │
│  ·  ·  ·  ·  ·  ·  ·   │   + corners checked explicitly
└─────────────────────────┘
```

The result is a `set[ChunkRef]` — deduplicated because multiple sample points can land in the same chunk.

**Corner check:** The loop at `spacing` intervals can miss chunks that *just barely* overlap a corner. The explicit corner sampling catches those.

### Step 3 — Stitch chunks (the heavy lift)

`stitch_chunks_fast` does:

1. For each `ChunkRef`, look up the cached zoomed chunk as a `DataProxy`
2. On cache miss, call `load_or_generate_chunk` (runs the actual zoom computation), then reload as `DataProxy`
3. Compute the merged grid dimensions from the range of coarse positions × scale
4. Copy each chunk's 2D elevation/watershed arrays into the merged array at its offset

The key equations for merged dimensions:

$$H_{\text{merged}} = (r_{\max}^{\text{coarse}} - r_{\min}^{\text{coarse}}) \times \text{scale} + H_{\text{chunk}}$$
$$W_{\text{merged}} = (c_{\max}^{\text{coarse}} - c_{\min}^{\text{coarse}}) \times \text{scale} + W_{\text{chunk}}$$

And the placement offset for each chunk:

$$\text{row\_off}^{(k)} = (r_k^{\text{coarse}} - r_{\min}^{\text{coarse}}) \times \text{scale}$$
$$\text{col\_off}^{(k)} = (c_k^{\text{coarse}} - c_{\min}^{\text{coarse}}) \times \text{scale}$$

This is all numpy 2D slice assignment — no per-hex loops.

### Step 4 — Build the mapper (on uncropped proxy)

This is the `build_fine_to_coarse_mapper` from the previous deep dive. The critical point: it's built on the **uncropped** merged proxy, because we need it in Step 5 before we know what to crop.

### Step 5 — Identify the region's fine hexes

This is the step that connects the coarse `HexRegion` to the fine grid. For every valid fine hex:

$$\text{coarse\_idx} = \text{mapper}(\text{fine\_idx})$$

If `coarse_idx ∈ region.hexes`, this fine hex "belongs" to the region.

```
Coarse region (shaded):          Fine merged grid:
┌──┬──┬──┬──┬──┐                 ┌──┬──┬──┬──┬──┬──┬──┬──┬──┬──┐
│  │  │░░│░░│  │   mapper        │  │  │  │  │▓▓│▓▓│▓▓│▓▓│  │  │
│  │  │░░│░░│  │   ←────────     │  │  │  │  │▓▓│▓▓│▓▓│▓▓│  │  │
│  │  │░░│  │  │                 │  │  │  │  │▓▓│▓▓│  │  │  │  │
│  │  │  │  │  │                 │  │  │  │  │  │  │  │  │  │  │
└──┴──┴──┴──┴──┘                 └──┴──┴──┴──┴──┴──┴──┴──┴──┴──┘

                                  ▓▓ = fine hexes where mapper → region hex
```

Three types of fine hexes get marked **invalid** here:
- Already in `invalidRegion` (from stitching)
- Elevation ≤ −99.0 (deep water sentinel)
- Mapper returns −1 (outside any chunk)

This is the most expensive loop — `O(merged_size)` — but it's pure Python integer ops, no geometry.

### Step 6 — Crop the proxy

The merged proxy can be much larger than the region. Imagine a region that's 5 coarse hexes in a vertical strip — the merged proxy might be 4 chunks wide but we only need a narrow column.

$$\text{crop\_padding} = \max(4,\; \text{scale} \times (\text{halo\_rings} + 1))$$

The crop finds the bounding box of `region_fine_indices`, pads it, and extracts a numpy 2D slice:

```
Before crop (merged proxy):      After crop:
┌────────────────────────┐        ┌──────────┐
│                        │        │          │
│       ┌──────┐         │   →    │ ┌──────┐ │
│       │region│         │        │ │region│ │
│       └──────┘         │        │ └──────┘ │
│                        │        │          │
└────────────────────────┘        └──────────┘
    50×80 = 4000 cells              20×30 = 600 cells
```

**Bail-out:** If cropping would save less than 10% of cells, we skip it (the overhead isn't worth it).

The function returns the crop offsets `(trim_top, trim_left)` needed to translate cropped coordinates back to merged coordinates.

### Step 7 — Wrap the mapper

Since the proxy was cropped, fine indices shifted. `make_cropped_mapper` wraps the original mapper:

$$\text{merged\_row} = \text{cropped\_row} + \text{trim\_top}$$
$$\text{merged\_col} = \text{cropped\_col} + \text{trim\_left}$$
$$\text{merged\_idx} = \text{merged\_row} \times W_{\text{old}} + \text{merged\_col}$$

Then feeds `merged_idx` to the original uncropped mapper. If no cropping happened (trim = 0), the original mapper is returned as-is.

### Step 8 — Build Terrain ONCE

`proxy_to_terrain(cropped_proxy)` is the **only** place where `HexGrid` geometry gets constructed. This is the payoff of the entire DataProxy design:

- `stitch_chunks_fast` → numpy only
- `crop_proxy_to_region` → numpy only
- Steps 5-7 → integer arithmetic only
- **Here** → create hex objects, compute pixel positions, etc.

We do it on the **cropped** proxy, so the HexGrid is as small as possible.

### Step 9 — Project weather (vectorized)

Weather lives on the coarse terrain. We need it on the fine terrain. The mapper gives us the correspondence, and we vectorize it:

```python
fine_arr   = [  0,   1,   2,   5,   6, ...]   # fine indices (valid)
coarse_arr = [ 42,  42,  42,  43,  43, ...]   # their coarse parents
```

For each weather field (temperature, precipitation, etc.):

$$\text{fine\_field}[\text{fine\_arr}] = \text{coarse\_field}[\text{coarse\_arr}]$$

This is a single numpy **fancy-index scatter** — no Python loop over hexes. Every fine hex in a `scale × scale` block gets the same coarse value (nearest-neighbor projection).

**Fallback:** If temperature or precipitation still isn't populated (e.g. the coarse terrain didn't have them), it calls `merged_terrain.compute_weather()` to generate from scratch.

### Step 10 — Project watersheds

We group the fine→coarse mapping into `fine_regions`:

$$\text{fine\_regions}[c] = \{f \;|\; \text{mapper}(f) = c\}$$

This is exactly the dict that `project_watersheds` expects — it assigns each fine hex the watershed ID of its coarse parent from the `DrainageBasins`.

The coarse `DrainageBasins` is lazily computed once (Step 3's `self.basin` check) and then reused for all subsequent zooms.

---

### Worked example

Setup:
- `cover` with `rings=3`, `halo_rings=1`, coarse grid `20×20`
- Region = 6 coarse hexes forming an L-shape
- `scale = 2`

```
Step 1:  bbox = rows [4,8], cols [6,9]
         padded = rows [2,10], cols [4,11]  (padding = 2)

Step 2:  At spacing=6, samples hit chunks (0,0,0) and (1,0,-1)
         → 2 ChunkRefs

Step 3:  Each chunk zooms to 14×14 fine hexes
         Merged proxy = 14×28 = 392 cells  (chunks side by side)

Step 4:  Mapper closure built (captures chunk_offsets, scale=2, etc.)

Step 5:  Loop over 392 fine cells:
         - 168 map to region hexes → region_fine_indices
         - 180 map to non-region hexes → ignored
         - 44 invalid (water/edge)

Step 6:  region_fine_indices bbox = rows [4,17], cols [4,17]
         Crop to 18×18 = 324 cells (saves 17%)
         trim_top=4, trim_left=4

Step 7:  Mapper wrapped: cropped_idx → +4 to both row,col → original mapper

Step 8:  HexGrid(18, 18, radius=5) built — 324 hexes with geometry

Step 9:  temperature[fine_arr] = coarse_temp[coarse_arr]  (one numpy op per field)

Step 10: fine_regions = {42: {0,1,2,3}, 43: {4,5,6,7}, ...}
         Each set gets its parent's watershed_id
         → ZoomResult with 168 meaningful fine hexes
```

### Performance characteristics

| Step | Cost | Dominates when... |
|------|------|-------------------|
| 3. Stitch | O(chunks × chunk_size) | Many chunks, cache cold |
| 5. Region filter | O(merged_size) | Large merged proxy |
| 8. Build Terrain | O(cropped_size) | Large region |
| 9. Weather | O(valid_fine × fields) | Many weather fields |

The cache-cold case (first zoom) is dominated by Step 3 — actually computing the zoomed terrain. Subsequent zooms hit the cache and Steps 5-9 dominate, which are all numpy-fast.

### The returned ZoomResult

```python
@dataclass
class ZoomResult:
    terrain: Terrain          # Full Terrain with HexGrid, elevations, weather fields
    basins: DrainageBasins    # Watersheds projected from coarse → fine
    mapper: Callable          # cropped_fine_idx → coarse_idx (for further lookups)
    chunks_loaded: int        # How many chunks were involved
```

The `mapper` is returned so callers can do their own fine→coarse lookups (e.g., "which biome does this fine hex inherit?" or "what political territory is it in?"). It's the cropped version, so it works directly with indices into `terrain.hexGrid`.

We can further use this to figure out where things are in detail

We now have
```python
@patch(as_prop=True)
def c2f(self: ZoomResult) -> dict:
    """Lazily compute and cache the coarse→fine index mapping."""
    if not hasattr(self, '_c2f') or self._c2f is None:
        self._c2f = self.invert_mapper()
    return self._c2f

@patch
def project_region(self: ZoomResult, region: HexRegion) -> HexRegion:
    """Project a coarse region onto the zoomed fine grid."""
    fine_hexes = set()
    for coarse_idx in region.hexes:
        if coarse_idx in self.c2f:
            fine_hexes.update(self.c2f[coarse_idx])
    return HexRegion(hexes=fine_hexes, hexGrid=self.terrain.hexGrid)

@patch
def map_point(self: ZoomResult, coarse_idx: int) -> int:
    """Map a single coarse index to a representative fine index (middle of mapped set)."""
    fs = self.c2f.get(coarse_idx)
    if not fs:
        return -1
    return fs[len(fs) // 2]
```

## Piece View — Project Plan
That's a really clean three-layer design:

1. **Kingdom Directive** — strategic intent ("we're starving", "explore", "expand")
2. **Piece Orders** — the directive gets mapped onto each piece based on its type and base stats
3. **Moves** — resolved outcomes after all kingdoms' orders collide

So the directive acts like the queen bee's pheromone signal — it shifts *what* each piece does, but *how well* they do it still depends on their base stats. A "gather food" directive would assign harvest orders to pawns (who are great at it) but might also reassign knights to patrol the borders instead of scouting.

For the directive enum, something like:

- **FEED** — prioritize harvest
- **EXPLORE** — prioritize scouting / vision
- **EXPAND** — claim new territory
- **DEFEND** — protect capital and settlements
- **ATTACK** — target enemy pieces/settlements

These look right. So we would to combine them to make a patrol concept. we could have a gosper curve that a piece would through as it moves. if we see something and it wants to attack then it could

That's a brilliant use of the Gosper curve — it's space-filling, so a piece on patrol would efficiently cover an area without a lot of backtracking. The concept would be something like:

    Generate patrol path — a Gosper curve anchored to a settlement or border, sized to fit the piece's territory
    Each turn — the piece advances along the curve by move_strength hexes
    At each hex — check sight range for triggers (enemy pieces, unclaimed resources, etc.)
    Interrupt — if something is spotted, the piece breaks off patrol and generates a new order (attack, harvest, scout closer)

So a patrol is really a streaming order — it keeps going until something more interesting happens. The directive level would control where the patrol is anchored and what triggers a break:

    DEFEND → patrol around capital, interrupt on enemy sighting → attack
    EXPLORE → patrol outward from borders, interrupt on discovery → report back
    FEED → patrol farmable land, interrupt on good hex → harvest


Nice. So let's sketch the Order shape. Something like:
Copied!

class OrderType(Enum):
    PATROL = "patrol"
    ATTACK = "attack"
    MOVE   = "move"
    HARVEST = "harvest"
    SCOUT  = "scout"




## Direction Picker

In [ ]:
#| export
# === Facing Selector Widget ===

_DIR_ARROWS = ["↙", "←", "↖", "↗", "→", "↘"]

def facing_grid(piece_id: str, current_facing: int = 0, radius: int = 30):
    """Build a 1-ring compass rose for selecting piece facing."""
    base   = StyleCSS("fc-hex",    fill="#f8f8f8", stroke="#999", stroke_width=1.5)
    active = StyleCSS("fc-active", fill="#4CAF50", stroke="#333", stroke_width=2)
    center = StyleCSS("fc-center", fill="#e0e0e0", stroke="#666", stroke_width=1.5)
    lbl    = StyleCSS("fc-label",  font_size="13px")

    grid = HexGrid.centered(1, radius=radius, style=base)
    for s in [active, center, lbl]:
        grid.builder.add_style(s)

    mid = grid.middle
    grid.hexes[mid].style = center
    grid.hexes[mid].label = "⬡"
    grid.hexes[mid].labelStyle = lbl.name

    # Build grid-index → facing-index map
    dir_map = {}
    for fi, d in enumerate(HexPosition.directions()):
        gi = grid.hexposition_to_index(d, mid)
        dir_map[gi] = fi
        grid.hexes[gi].label = _DIR_ARROWS[fi]
        grid.hexes[gi].labelStyle = lbl.name
        if fi == current_facing:
            grid.hexes[gi].style = active

    grid.update()

    target_id = f"facing-{piece_id[:8]}"

    def on_click(g, i):
        if i in dir_map:
            return {
                "hx-post":   f"/set_facing/{piece_id}/{dir_map[i]}",
                "hx-target": f"#{target_id}",
                "hx-swap":   "outerHTML",
            }
        return {}

    return Div(
        H5("Facing", cls="font-bold text-sm mb-1"),
        HexTouchMap(grid, on_click=on_click),
        id=target_id,
        cls="inline-block"
    )


@rt("/facing_selector/{piece_id}")
def facing_selector(session, piece_id: str):
    """Render the facing compass for a piece."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active: return P("No active game")
    piece = next((p for p in active.board.pieces if p.id == piece_id), None)
    if not piece: return P("Piece not found")
    return facing_grid(piece_id, current_facing=piece.facing)


@rt("/set_facing/{piece_id}/{facing:int}")
def set_facing(session, piece_id: str, facing: int):
    """Update piece facing and return refreshed compass."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active: return P("No active game")
    piece = next((p for p in active.board.pieces if p.id == piece_id), None)
    if not piece: return P("Piece not found")
    piece.facing = facing % 6
    return facing_grid(piece_id, current_facing=piece.facing)


## Getting Data

In [ ]:
showUsers()

In [ ]:
dummySession= {'userid': 64801 }

In [ ]:
#globalStore.create_game(dummySession['userid'], template_name="bayArea")

Can we add this dummy user and then add a Knight piece to the first settlement in the first kingom of its active world

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Sure! Here's the code to do all three steps:

```python
from datetime import datetime

# 1. Add the dummy user
now = int(datetime.now().timestamp())
globalStore.users.insert(dict(
    username="dummy", email="dummy@test.com", password="test",
    created=now, sessionID="", activeWorld=0, id=64801
))

# 2. Create a world for them
active = globalStore.create_game(64801, template_name="bayArea")

# 3. Recruit a Knight into the first settlement of the first kingdom
board = active.board
kingdom = board.kingdoms[0]
settlement = kingdom.settlements[0]
settlement.db = globalStore  # wire up DB so save works

knight = settlement.recruit(PieceType.KNIGHT, flag=kingdom.flag)
knight.save(db=globalStore, world_id=active.world_id, kingdom_id=kingdom.countryId)

print(f"User: 64801")
print(f"World: {active.world_id}")
print(f"Kingdom: {kingdom.countryName} (id={kingdom.countryId})")
print(f"Settlement: {settlement.name} @ hex {settlement.location}")
print(f"Knight: {knight.id[:12]}… facing={knight.facing} atk={knight.attack_strength} mov={knight.move_strength}")
```

This will:
1. Insert the dummy user with id `64801`
2. Generate a "bayArea" world with expanded kingdoms and persist it
3. Recruit a `KNIGHT` into the first kingdom's capital settlement — the knight gets the default knight stats `(atk=2, mov=7, harvest=1, scout=5)` and the kingdom's flag — then saves it to the DB

In [ ]:
def addMissing():
    from datetime import datetime

    # 1. Add the dummy user
    now = int(datetime.now().timestamp())
    globalStore.users.insert(dict(
        username="dummy", email="dummy@test.com", password="test",
        created=now, sessionID="", activeWorld=0, id=64801
    ))

    # 2. Create a world for them
    active = globalStore.create_game(64801, template_name="bayArea")

    # 3. Recruit a Knight into the first settlement of the first kingdom
    board = active.board
    kingdom = board.kingdoms[0]
    settlement = kingdom.settlements[0]
    settlement.db = globalStore  # wire up DB so save works

    knight = settlement.recruit(PieceType.KNIGHT, flag=kingdom.flag)
    knight.save(db=globalStore, world_id=active.world_id, kingdom_id=kingdom.countryId)

    print(f"User: 64801")
    print(f"World: {active.world_id}")
    print(f"Kingdom: {kingdom.countryName} (id={kingdom.countryId})")
    print(f"Settlement: {settlement.name} @ hex {settlement.location}")
    print(f"Knight: {knight.id[:12]}… facing={knight.facing} atk={knight.attack_strength} mov={knight.move_strength}")


In [ ]:
addMissing()

In [ ]:
def showPieces():
    users_df = pd.DataFrame(globalStore.pieces())
    print("pieces:")
    print(users_df)

In [ ]:
showPieces()

In [ ]:
def showActiveSettlements(session):
    uid = ensure_user(session)
    logging.info(f"showSettlement: uid={uid}")
    active = globalStore.active_board(uid)
    ret = []
    for country in active.board.kingdoms:
        ret.extend( country.settlements)
    return ret


In [ ]:
showActiveSettlements(dummySession)

In [ ]:
def listSettlements():
    users_df = pd.DataFrame(globalStore.settlements())
    print("pieces:")
    print(users_df)

In [ ]:
listSettlements()

In [ ]:
dummySettlment = "7ba2cc4b-97fd-4682-8d34-e4b9ea06cd72"

In [ ]:
@rt("/piece_map/{id}")
def piece_map(session, id: str, rings: int = None):
    """Map partial — renders zoomed terrain with active overlays."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return P("No active game", id="map")

    if rings is not None:
        rings = max(3, min(12, rings))
        session['settlement_rings'] = rings
    else:
        rings = session.get('settlement_rings', 5)

    overlays = get_overlays(session)

    try:
        place = globalStore.piece_from_id(id)
        board = active.board
        terrain = board.terrain

        region = place.region(terrain.hexGrid, rings=rings)
        if len(region.hexes) == 0:
            return P("Settlement region is empty", id="map")

        result = active.cover.zoom_region_fast(region, compute_weather=True)
        zoomed = result.terrain
        zoomed.hexGrid.adjustRadius(terrain.hexGrid.radius)
        zoomed.hexGrid.builder.layers = []
        zoomed.colorMap()
        zoomed.hexGrid.update()
        c2f = result.invert_mapper()

        piece_attrs = {
            'hx-get': f'/piece_detail/{id}',
            'hx-target': '#right-panel',
            'hx-swap': 'innerHTML',
            'style': 'cursor: pointer;',
        }
        zoomed.apply_overlays(result, board, c2f, overlays, piece_attrs)

        return Div(
            HexTouchMap(zoomed.hexGrid, cls="w-full h-full"),
            id="map"
        )
    except Exception as e:
        logging.error(f"settlement_map FAILED: {e}", exc_info=True)
        return P(f"Error: {e}", id="map")


Piece needs to have an overlay. lets start building the one.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good idea! Looking at the existing overlays (`countries_overlay`, `names_overlay`, `settlementOverlay`), they all follow the same pattern:

1. Loop over kingdoms → settlements → citizens
2. Map each piece's `location` through `c2f` (for zoomed views) via `_map_point`
3. Look up pixel coords from `grid.hexes[local_idx].center`
4. Emit SVG at those coords

The key decision is: **what visual should represent each piece type?** The settlement overlay uses `country.flag.kingPiece()` which draws a crown-like SVG shape. For pieces you'll need something for each `PieceType` (PAWN, KNIGHT, ROOK, BISHOP, QUEEN, KING) plus a facing indicator (the `facing` 0–5 maps to a direction).

A simple starting point would be a small circle at the piece's location with an arrow/line showing facing direction, colored by the kingdom's flag. Then you can upgrade to distinct shapes per type later.

Does that sound right? And would you like to put this as a `@patch` on `GameBoard` (like `settlementOverlay`), or somewhere else?

So I am going to start having the flag be what marks settlments so the king returns (good book). So the idea is for a piece map that would show what the piece would do. we should have a number of rounds to show where the piece thinks is going to happen based upon its rules. We are going to want to have that for a group of pieces as well when we have the settlement and kindom maps. We need a region large enough to show all of the moves. so we will have to find the hexes in the fine region and then use the zoom-region to get them which is a smidge complicated

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Right, so the flow would be something like:

1. **Simulate** — walk the piece's `rules` for N rounds from its current `location`/`facing`, collecting the hex positions it would visit (on the coarse grid)
2. **Compute region** — take all those coarse hexes + some padding, build a `HexRegion`
3. **Zoom** — `zoom_region_fast` on that region to get fine terrain
4. **Project & draw** — map each coarse path hex through `c2f` to fine coords, render as a styled path overlay (arrows, numbered steps, etc.)

The tricky part you're flagging is step 2→3 — the region has to be big enough for the *full* simulated path, not just the piece's current position. And for a group of pieces (settlement/kingdom view), you'd union all their projected paths to get the bounding region before zooming.

I'd start with the simulation — a method on `Piece` that takes a `HexGrid` and `num_rounds`, steps through `rules` respecting `cursor`/`patrol`/`facing`, and returns a list of `(hex_index, facing)` tuples for each round. That becomes the foundation for everything else.

Want to start with that simulate method?

I guess we need a few parts on the middle where we get a large patch of terrain because we have to see if the piece deadends in water and has to turn to figure out the hexes. then use that as the basis for the region. The second time we should be hitting a cache. Though it might be easier to just get the elevations instead of the full result.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good thinking, but actually — you can sidestep the chicken-and-egg entirely. **Simulate on the coarse grid**, which you already have loaded:

- `Piece.location` is a coarse grid index
- `terrain.elevations[location]` tells you water (< 1) or impassable mountain
- `HexGrid` + `HexPosition.directions()` gives you neighbors for movement

So the pipeline becomes:

```
Coarse grid (already loaded in ActiveGame)
    │
    ▼
1. Simulate on coarse grid  ← uses elevations + hexGrid only
   → List[(hex_idx, facing)] per round
    │
    ▼
2. Collect all visited hex_idx + padding → HexRegion
    │
    ▼
3. zoom_region_fast(region)  ← chunk cache likely warm
   → ZoomResult
    │
    ▼
4. Project coarse path through c2f → draw on fine grid
```

Step 1 is pure integer arithmetic on the coarse grid — no zoom needed. When the piece hits water, it knows immediately from `elevations[neighbor_idx] < 1` and can apply its turn logic (ROT_L / ROT_R from the `Instruction` enum).

Step 3 will mostly cache-hit if the player has looked at this area before.

So the simulate method signature would be something like:

```python
@patch
def simulate(self: Piece, grid: HexGrid, elevations: np.ndarray, 
             num_rounds: int = 10) -> list[tuple[int, int]]:
    """Walk rules for N rounds on coarse grid.
    Returns [(hex_idx, facing)] for each step, 
    stopping early if stuck."""
```

And the logic inside would step through `self.rules[self.cursor]`, applying each `Instruction`:
- **FORWARD** → compute neighbor in `facing` direction, check elevation, move or skip
- **ROT_L/ROT_R** → update facing
- **PAUSE/DEFEND/HARVEST/GIVE/SETTLE** → stay in place (hex stays same, facing stays same)
- When cursor hits end of rules, wrap if `patrol=True`, else stop

Want to build that simulate method first?

so we are going to want a datastructue to draw the piece future movement 
def pieceMove
    startIndex
    endIndex
    cost # in case we need to draw a swiggle
    stopKind = harvest, give, defend.

    but this is really the rules list anyways. we just need to append the approaite svg to a str

Do you think you can build simulare, the overlay and a demo?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here are three blocks — simulation core, overlay renderer, and demo.

**Block 1: PieceStep + simulate + plan_region**

```python
@dataclass
class PieceStep:
    """One resolved step in a piece's movement plan."""
    hex_idx: int
    facing: int
    instruction: Instruction
    prev_idx: int
    step: int
    blocked: bool = False

@patch
def simulate(self: Piece, grid: HexGrid, elevations: np.ndarray,
             num_rounds: int = 10, min_elev: float = 1.0) -> list['PieceStep']:
    """Walk rules on coarse grid for N steps."""
    if not self.rules or self.location is None: return []
    steps = []
    pos, fac, cur = self.location, self.facing, self.cursor

    for n in range(num_rounds):
        if cur >= len(self.rules):
            if self.patrol: cur = 0
            else: break

        instr = Instruction(self.rules[cur])
        prev, blocked = pos, False

        if instr == Instruction.FORWARD:
            d = HexPosition.directions()[fac % 6]
            nbr = grid.hexposition_to_index(d, pos)
            if (0 <= nbr < len(elevations)
                    and nbr not in grid.invalidRegion
                    and elevations[nbr] >= min_elev):
                pos = nbr
            else:
                blocked = True
        elif instr == Instruction.ROT_L:
            fac = (fac - 1) % 6
        elif instr == Instruction.ROT_R:
            fac = (fac + 1) % 6
        # PAUSE / DEFEND / HARVEST / GIVE / SETTLE → stay in place

        steps.append(PieceStep(hex_idx=pos, facing=fac, instruction=instr,
                               prev_idx=prev, step=n, blocked=blocked))
        cur += 1
    return steps

@patch
def plan_region(self: Piece, grid: HexGrid, elevations: np.ndarray,
                num_rounds: int = 10, padding: int = 3
                ) -> tuple[list['PieceStep'], 'HexRegion']:
    """Simulate and return (steps, bounding_region) for zooming."""
    steps = self.simulate(grid, elevations, num_rounds)
    visited = {self.location}
    for s in steps: visited.add(s.hex_idx)
    expanded = set()
    for h in visited:
        for n in grid.indices_in_range(h, padding):
            if 0 <= n < len(elevations): expanded.add(n)
    return steps, HexRegion(hexes=expanded, hexGrid=grid)
```

**Block 2: Overlay renderer**

```python
_STEP_GLYPHS = {
    Instruction.PAUSE:   "⏸",
    Instruction.DEFEND:  "🛡",
    Instruction.HARVEST: "🌾",
    Instruction.GIVE:    "🤝",
    Instruction.SETTLE:  "🏠",
    Instruction.ROT_L:   "↺",
    Instruction.ROT_R:   "↻",
}

def _map_pt(coarse_idx, c2f=None):
    """Map coarse index to fine (or identity if no c2f)."""
    if c2f is None: return coarse_idx
    fs = c2f.get(coarse_idx)
    return fs[len(fs)//2] if fs else -1

def piece_plan_overlay(steps: list[PieceStep], grid: HexGrid,
                       color: str = "#4CAF50", c2f: dict = None,
                       start_idx: int = -1) -> str:
    """Render plan as SVG: dashed arrows for moves, glyphs for actions."""
    svg, N = "", len(grid.hexes)

    # Start ring
    si = _map_pt(start_idx, c2f) if start_idx >= 0 else -1
    if 0 <= si < N:
        sx, sy = grid.hexes[si].center.x, grid.hexes[si].center.y
        svg += (f'\t<circle cx="{sx:.1f}" cy="{sy:.1f}" r="7" '
                f'fill="none" stroke="{color}" stroke-width="2.5" opacity="0.9"/>\n')

    for step in steps:
        idx  = _map_pt(step.hex_idx, c2f)
        prev = _map_pt(step.prev_idx, c2f) if step.prev_idx >= 0 else -1
        if idx < 0 or idx >= N: continue
        cx, cy = grid.hexes[idx].center.x, grid.hexes[idx].center.y

        # Movement line + dot
        if (step.instruction == Instruction.FORWARD and not step.blocked
                and 0 <= prev < N and prev != idx):
            px, py = grid.hexes[prev].center.x, grid.hexes[prev].center.y
            svg += (f'\t<line x1="{px:.1f}" y1="{py:.1f}" x2="{cx:.1f}" y2="{cy:.1f}" '
                    f'stroke="{color}" stroke-width="2" '
                    f'stroke-dasharray="4,2" opacity="0.7"/>\n')
            svg += (f'\t<circle cx="{cx:.1f}" cy="{cy:.1f}" r="3.5" '
                    f'fill="{color}" opacity="0.6"/>\n')

        # Blocked
        elif step.blocked:
            svg += (f'\t<circle cx="{cx:.1f}" cy="{cy:.1f}" r="5" '
                    f'fill="red" stroke="darkred" stroke-width="1" opacity="0.5"/>\n')

        # Action glyph
        else:
            glyph = _STEP_GLYPHS.get(step.instruction, "")
            if glyph:
                svg += (f'\t<text x="{cx:.1f}" y="{cy:.1f}" text-anchor="middle" '
                        f'dominant-baseline="middle" font-size="13">{glyph}</text>\n')

        # Step number
        svg += (f'\t<text x="{cx+7:.1f}" y="{cy-7:.1f}" text-anchor="start" '
                f'font-size="8" fill="#555" font-weight="bold">{step.step+1}</text>\n')

    return svg
```

**Block 3: Demo**

```python
uid = ensure_user(dummySession)
active = globalStore.active_board(uid)
board, terrain, grid = active.board, active.board.terrain, active.board.terrain.hexGrid

# Find the existing piece (the knight)
kingdom = next(k for k in board.kingdoms
               if any(c for s in k.settlements for c in s.citizens))
piece = next(c for s in kingdom.settlements for c in s.citizens)

# Assign a patrol route: forward-forward-turn, with a harvest stop
piece.rules = [
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.HARVEST.value,
    Instruction.FORWARD.value,
    Instruction.ROT_L.value,
]
piece.patrol = True

# Simulate 20 steps on the coarse grid (cheap — just integer lookups)
steps, region = piece.plan_region(grid, terrain.elevations,
                                  num_rounds=20, padding=4)

print(f"Piece: {piece.piece_type.name} at hex {piece.location}, facing={piece.facing}")
print(f"{len(steps)} steps, {len(region)} hex region\n")
for s in steps:
    tag = " ✘" if s.blocked else ""
    print(f"  {s.step+1:2d}. {s.instruction.name:8s} → hex {s.hex_idx}  f={s.facing}{tag}")

# Zoom into the plan region (chunk cache likely warm)
result = active.cover.zoom_region_fast(region, compute_weather=True)
zoomed = result.terrain
zoomed.hexGrid.adjustRadius(grid.radius)
zoomed.colorMap()
zoomed.hexGrid.update()

# Draw the plan overlay on the zoomed grid
overlay = piece_plan_overlay(steps, zoomed.hexGrid,
                             color="#e74c3c", c2f=result.c2f,
                             start_idx=piece.location)
zoomed.hexGrid.builder.adjust("plan", overlay)

from IPython.display import SVG, display
display(SVG(zoomed.hexGrid.builder.xml()))
```

The flow: `simulate` walks the `rules` list on the **coarse grid** (instant — just array lookups), collecting every hex the piece visits. `plan_region` pads those hexes to build a viewing region. `zoom_region_fast` generates fine terrain for that region (cache-warm on repeat). Then `piece_plan_overlay` maps each step through `c2f` onto the fine grid and emits SVG — dashed lines for moves, emoji glyphs for actions, red dots for blocked moves, and step numbers.

For a group of pieces (settlement/kingdom view), you'd union all their `plan_region` hexes before the single `zoom_region_fast` call, then draw each piece's overlay in a different color.

In [ ]:
#| export
@dataclass
class PieceStep:
    """One resolved step in a piece's movement plan."""
    hex_idx: int
    facing: int
    instruction: Instruction
    prev_idx: int
    step: int
    blocked: bool = False




In [ ]:
#| export
@patch
def simulate(self: Piece, grid: HexGrid, elevations: np.ndarray,
             num_rounds: int = 10, min_elev: float = 1.0,
             countries: np.ndarray = None) -> list['PieceStep']:
    """Walk rules on coarse grid for N steps."""
    if not self.rules or self.location is None: return []
    steps = []
    pos, fac, cur = self.location, self.facing, self.cursor

    for n in range(num_rounds):
        if cur >= len(self.rules):
            if self.patrol: cur = 0
            else: break

        instr = Instruction(self.rules[cur])
        prev, blocked = pos, False

        if instr == Instruction.FORWARD:
            d = HexPosition.directions()[fac % 6]
            nbr = grid.hexposition_to_index(d, pos)
            passable = (0 <= nbr < len(elevations)
                        and nbr not in grid.invalidRegion
                        and elevations[nbr] >= min_elev)
            # Also block if country field marks it as water/mountain (-1)
            if passable and countries is not None:
                passable = countries[nbr] >= 0
            if passable:
                pos = nbr
            else:
                blocked = True
        elif instr == Instruction.ROT_L:
            fac = (fac - 1) % 6
        elif instr == Instruction.ROT_R:
            fac = (fac + 1) % 6

        steps.append(PieceStep(hex_idx=pos, facing=fac, instruction=instr,
                               prev_idx=prev, step=n, blocked=blocked))
        cur += 1
    return steps

@patch
def plan_region(self: Piece, grid: HexGrid, elevations: np.ndarray,
                num_rounds: int = 10, padding: int = 3,
                countries: np.ndarray = None
                ) -> tuple[list['PieceStep'], 'HexRegion']:
    """Simulate and return (steps, bounding_region) for zooming."""
    steps = self.simulate(grid, elevations, num_rounds, countries=countries)
    visited = {self.location}
    for s in steps: visited.add(s.hex_idx)
    expanded = set()
    for h in visited:
        for n in grid.indices_in_range(h, padding):
            if 0 <= n < len(elevations): expanded.add(n)
    return steps, HexRegion(hexes=expanded, hexGrid=grid)


def piece_plan_overlay(steps: list[PieceStep], grid: HexGrid,
                       color: str = "#4CAF50", c2f: dict = None,
                       start_idx: int = -1) -> str:
    """Render plan as SVG: dashed arrows for moves, glyphs for actions."""
    svg, N = "", len(grid.hexes)

    # Start ring
    si = _map_pt(start_idx, c2f) if start_idx >= 0 else -1
    if 0 <= si < N:
        sx, sy = grid.hexes[si].center.x, grid.hexes[si].center.y
        svg += (f'\t<circle cx="{sx:.1f}" cy="{sy:.1f}" r="7" '
                f'fill="none" stroke="{color}" stroke-width="2.5" opacity="0.9"/>\n')

    for step in steps:
        idx  = _map_pt(step.hex_idx, c2f)
        prev = _map_pt(step.prev_idx, c2f) if step.prev_idx >= 0 else -1
        if idx < 0 or idx >= N: continue
        cx, cy = grid.hexes[idx].center.x, grid.hexes[idx].center.y

        # Movement line + dot
        if (step.instruction == Instruction.FORWARD and not step.blocked
                and 0 <= prev < N and prev != idx):
            px, py = grid.hexes[prev].center.x, grid.hexes[prev].center.y
            svg += (f'\t<line x1="{px:.1f}" y1="{py:.1f}" x2="{cx:.1f}" y2="{cy:.1f}" '
                    f'stroke="{color}" stroke-width="2" '
                    f'stroke-dasharray="4,2" opacity="0.7"/>\n')
            svg += (f'\t<circle cx="{cx:.1f}" cy="{cy:.1f}" r="3.5" '
                    f'fill="{color}" opacity="0.6"/>\n')

        # Blocked
        elif step.blocked:
            svg += (f'\t<circle cx="{cx:.1f}" cy="{cy:.1f}" r="5" '
                    f'fill="red" stroke="darkred" stroke-width="1" opacity="0.5"/>\n')

        # Action glyph
        else:
            glyph = _STEP_GLYPHS.get(step.instruction, "")
            if glyph:
                svg += (f'\t<text x="{cx:.1f}" y="{cy:.1f}" text-anchor="middle" '
                        f'dominant-baseline="middle" font-size="13">{glyph}</text>\n')

        # Step number
        svg += (f'\t<text x="{cx+7:.1f}" y="{cy-7:.1f}" text-anchor="start" '
                f'font-size="8" fill="#555" font-weight="bold">{step.step+1}</text>\n')

    return svg

In [ ]:
HexGrid.arrow??

In [ ]:
MapPath.make_windy??

In [ ]:
CountryFlag.knightPiece??

In [ ]:
_PIECE_RENDERERS = {
    PieceType.PAWN:   'pawnPiece',
    PieceType.KNIGHT: 'knightPiece',
    PieceType.ROOK:   'rookPiece',
    PieceType.BISHOP: 'bishopPiece',
    PieceType.QUEEN:  'queenPiece',
    PieceType.KING:   'kingPiece',
}

_STEP_GLYPHS = {
    Instruction.PAUSE:   "⏸",
    Instruction.DEFEND:  "🛡",
    Instruction.HARVEST: "🌾",
    Instruction.GIVE:    "🤝",
    Instruction.SETTLE:  "🏠",
    Instruction.ROT_L:   "↺",
    Instruction.ROT_R:   "↻",
}

def piece_plan_overlay(steps: list[PieceStep], grid: HexGrid,
                       piece: Piece = None, c2f: dict = None,
                       color: str = "#4CAF50", move_scale: float = 1.5,
                       piece_scale: float = 0.6) -> str:
    """Render movement plan using grid arrows and flag chess pieces."""
    svg, N = "", len(grid.hexes)
    flag = piece.flag if piece else None

    # Arrow style for movement lines
    arrow_style = StyleCSS(f"plan_arrow_{color.strip('#')}",
                           stroke=color, stroke_width=2,
                           fill="none", opacity="0.7",
                           stroke_dasharray="6,3")
    grid.builder.add_style(arrow_style)

    # Draw the piece SVG at its start location
    start_fine = _map_point(piece.location, c2f) if piece and piece.location is not None else -1
    if 0 <= start_fine < N and flag:
        center = grid.hexes[start_fine].center
        renderer = _PIECE_RENDERERS.get(piece.piece_type, 'pawnPiece')
        svg += getattr(flag, renderer)(
            MapCord(center.x, center.y),
            scale=piece_scale,
            piece_id=f"plan_start_{piece.id[:8]}",
        )

    for step in steps:
        fine_idx  = _map_point(step.hex_idx, c2f)
        fine_prev = _map_point(step.prev_idx, c2f) if step.prev_idx >= 0 else -1
        if fine_idx < 0 or fine_idx >= N:
            continue
        cx, cy = grid.hexes[fine_idx].center.x, grid.hexes[fine_idx].center.y

        # === FORWARD move → spline arrow ===
        if (step.instruction == Instruction.FORWARD and not step.blocked
                and 0 <= fine_prev < N and fine_prev != fine_idx):
            svg += grid.arrow(fine_prev, fine_idx, style=arrow_style, factor=0.3)

        # === Blocked move → red X ===
        elif step.blocked:
            svg += (f'\t<circle cx="{cx:.1f}" cy="{cy:.1f}" r="5" '
                    f'fill="red" stroke="darkred" stroke-width="1" opacity="0.5"/>\n')
            svg += (f'\t<text x="{cx:.1f}" y="{cy+1:.1f}" text-anchor="middle" '
                    f'dominant-baseline="middle" font-size="9" fill="white" '
                    f'font-weight="bold">✘</text>\n')

        # === Action glyph (harvest, defend, etc.) ===
        else:
            glyph = _STEP_GLYPHS.get(step.instruction, "")
            if glyph:
                svg += (f'\t<text x="{cx:.1f}" y="{cy:.1f}" text-anchor="middle" '
                        f'dominant-baseline="middle" font-size="14">{glyph}</text>\n')

        # Step number label (offset up-right)
        svg += (f'\t<text x="{cx+8:.1f}" y="{cy-8:.1f}" text-anchor="start" '
                f'font-size="8" fill="#555" font-weight="bold">{step.step+1}</text>\n')

    # Ghost piece at final position (smaller, semi-transparent)
    if steps and flag:
        last = steps[-1]
        final_fine = _map_point(last.hex_idx, c2f)
        if 0 <= final_fine < N and final_fine != start_fine:
            end_center = grid.hexes[final_fine].center
            renderer = _PIECE_RENDERERS.get(piece.piece_type, 'pawnPiece')
            svg += getattr(flag, renderer)(
                MapCord(end_center.x, end_center.y),
                scale=piece_scale * 0.7,
                piece_id=f"plan_end_{piece.id[:8]}",
                attrs={"opacity": "0.4"},
            )

    return svg


In [ ]:
_STEP_GLYPHS = {
    Instruction.PAUSE:   "⏸",
    Instruction.DEFEND:  "🛡",
    Instruction.HARVEST: "🌾",
    Instruction.GIVE:    "🤝",
    Instruction.SETTLE:  "🏠",
    Instruction.ROT_L:   "↺",
    Instruction.ROT_R:   "↻",
}

def _map_pt(coarse_idx, c2f=None):
    """Map coarse index to fine (or identity if no c2f)."""
    if c2f is None: return coarse_idx
    fs = c2f.get(coarse_idx)
    return fs[len(fs)//2] if fs else -1




In [ ]:
hexCount = 16
radius = 80
padding = 20
itemWidth = (radius * 2 + padding)
cols = 6
rowHeight = radius * 2 + 60

canvas = SVGBuilder()
canvas.width = cols * itemWidth
canvas.height = 3 * rowHeight + padding

pattern_names = [
    "circle", "tri", "swirl", "yin", "weave",
    "sheridan", "chevron", "scales", "diamonds", "fanBlade", "leafy",
    "bicolorH", "bicolorV", "tricolorH", "quarters", "crossFlag"
]

flag = CountryFlag.seaborn("husl", 1)[0]

# Background rects so the flag pattern shows behind the king
for i in range(hexCount):
    flag.patternIndex = i
    name = f"gallery_{i}"
    patternName = f"{name}_pat"
    
    row = i // cols
    col = i % cols
    
    pat = flag.flagPattern(patternName,scale=0.1)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    cx = (padding + radius) + col * itemWidth
    cy = padding + radius + row * rowHeight
    
    # Draw a colored circle as backdrop, then the king on top
    king_scale = 3.0
    backdrop = f'<circle cx="{cx}" cy="{cy}" r="{radius * 0.7}" fill="white" stroke="black" stoke-width="1"/>'
    pat_fill = f"url(#{patternName})"
    king_svg = flag.knightPiece(MapCord(cx, cy), scale=king_scale, piece_id=f"king_{i}", fill=pat_fill)

    
    canvas.adjust(f"backdrop-{i}", backdrop)
    canvas.adjust(f"king-{i}", king_svg)
    canvas.add_style(style)
    canvas.add_definition(pat)
    
    label = f'<text x="{cx}" y="{cy + radius + 20}" text-anchor="middle" font-size="14" font-family="sans-serif" fill="#333">{pattern_names[i]}</text>'
    canvas.adjust(f"label-{i}", label)

canvas.show()


In [ ]:
def svg_circle(cx, cy, r, fill="none", stroke="#333", stroke_width=1.5, opacity=1.0, **kw):
    """Filled or outlined circle."""
    return (f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{r:.1f}" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="{stroke_width}" '
            f'opacity="{opacity}"/>\n')

def svg_ring(cx, cy, r, stroke, stroke_width=2, opacity=0.9):
    """Hollow ring (start marker, selection, etc.)."""
    return svg_circle(cx, cy, r, fill="none", stroke=stroke,
                      stroke_width=stroke_width, opacity=opacity)

def svg_arc(cx, cy, r, start_deg, end_deg, stroke, stroke_width=2, opacity=0.8):
    """Arc segment — good for partial rings, rotation indicators."""
    import math
    s = math.radians(start_deg)
    e = math.radians(end_deg)
    x1, y1 = cx + r * math.cos(s), cy + r * math.sin(s)
    x2, y2 = cx + r * math.cos(e), cy + r * math.sin(e)
    large = 1 if (end_deg - start_deg) % 360 > 180 else 0
    return (f'<path d="M {x1:.1f},{y1:.1f} A {r:.1f},{r:.1f} 0 {large} 1 {x2:.1f},{y2:.1f}" '
            f'fill="none" stroke="{stroke}" stroke-width="{stroke_width}" '
            f'opacity="{opacity}" stroke-linecap="round"/>\n')

def svg_shield(cx, cy, size, fill, stroke, opacity=0.85):
    """Shield shape for DEFEND — classic heraldic proportions."""
    w, h = size, size * 1.2
    return (f'<path d="M {cx:.1f},{cy-h*0.5:.1f} '
            f'L {cx+w*0.5:.1f},{cy-h*0.5:.1f} '
            f'L {cx+w*0.5:.1f},{cy+h*0.1:.1f} '
            f'Q {cx:.1f},{cy+h*0.5:.1f} {cx-w*0.5:.1f},{cy+h*0.1:.1f} '
            f'L {cx-w*0.5:.1f},{cy-h*0.5:.1f} Z" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')

def svg_sheaf(cx, cy, size, fill, stroke, opacity=0.85):
    """Wheat sheaf for HARVEST — three lines bound in middle."""
    s = size
    svg = ""
    # Three stalks
    for dx in [-s*0.3, 0, s*0.3]:
        svg += (f'<line x1="{cx+dx:.1f}" y1="{cy-s*0.5:.1f}" '
                f'x2="{cx+dx*0.3:.1f}" y2="{cy+s*0.5:.1f}" '
                f'stroke="{fill}" stroke-width="{max(1.5, s*0.15):.1f}" '
                f'stroke-linecap="round" opacity="{opacity}"/>\n')
    # Binding band
    svg += (f'<line x1="{cx-s*0.35:.1f}" y1="{cy:.1f}" '
            f'x2="{cx+s*0.35:.1f}" y2="{cy:.1f}" '
            f'stroke="{stroke}" stroke-width="{max(1, s*0.1):.1f}" opacity="{opacity}"/>\n')
    return svg

def svg_gift(cx, cy, size, fill, stroke, opacity=0.85):
    """Simple box with ribbon for GIVE."""
    s = size * 0.8
    svg = (f'<rect x="{cx-s*0.5:.1f}" y="{cy-s*0.4:.1f}" '
           f'width="{s:.1f}" height="{s*0.8:.1f}" '
           f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
           f'rx="{s*0.1:.1f}" opacity="{opacity}"/>\n')
    # Ribbon cross
    svg += (f'<line x1="{cx:.1f}" y1="{cy-s*0.4:.1f}" '
            f'x2="{cx:.1f}" y2="{cy+s*0.4:.1f}" '
            f'stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    svg += (f'<line x1="{cx-s*0.5:.1f}" y1="{cy:.1f}" '
            f'x2="{cx+s*0.5:.1f}" y2="{cy:.1f}" '
            f'stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    return svg

def svg_house(cx, cy, size, fill, stroke, opacity=0.85):
    """Simple house for SETTLE."""
    s = size
    # Walls
    svg = (f'<rect x="{cx-s*0.4:.1f}" y="{cy:.1f}" '
           f'width="{s*0.8:.1f}" height="{s*0.5:.1f}" '
           f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    # Roof
    svg += (f'<polygon points="'
            f'{cx-s*0.5:.1f},{cy:.1f} '
            f'{cx:.1f},{cy-s*0.45:.1f} '
            f'{cx+s*0.5:.1f},{cy:.1f}" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    return svg

def svg_pause(cx, cy, size, fill, stroke, opacity=0.7):
    """Two vertical bars for PAUSE."""
    w, h = size * 0.2, size * 0.6
    for dx in [-size*0.18, size*0.18]:
        fill_c = (f'<rect x="{cx+dx-w/2:.1f}" y="{cy-h/2:.1f}" '
                  f'width="{w:.1f}" height="{h:.1f}" '
                  f'fill="{fill}" stroke="{stroke}" stroke-width="1" '
                  f'rx="{w*0.2:.1f}" opacity="{opacity}"/>\n')
        yield fill_c

def svg_rot_arrow(cx, cy, r, clockwise: bool, stroke, stroke_width=2, opacity=0.85):
    """Circular arc arrow for ROT_L / ROT_R."""
    # 270° arc + arrowhead
    if clockwise:
        arc = svg_arc(cx, cy, r, -120, 150, stroke, stroke_width, opacity)
        # Arrowhead tip at end_deg=150
        import math
        tip_x = cx + r * math.cos(math.radians(150))
        tip_y = cy + r * math.sin(math.radians(150))
        ah = (f'<polygon points="'
              f'{tip_x:.1f},{tip_y:.1f} '
              f'{tip_x-r*0.25:.1f},{tip_y-r*0.15:.1f} '
              f'{tip_x-r*0.1:.1f},{tip_y+r*0.2:.1f}" '
              f'fill="{stroke}" opacity="{opacity}"/>\n')
    else:
        arc = svg_arc(cx, cy, r, 30, 300, stroke, stroke_width, opacity)
        import math
        tip_x = cx + r * math.cos(math.radians(30))
        tip_y = cy + r * math.sin(math.radians(30))
        ah = (f'<polygon points="'
              f'{tip_x:.1f},{tip_y:.1f} '
              f'{tip_x+r*0.25:.1f},{tip_y-r*0.15:.1f} '
              f'{tip_x+r*0.1:.1f},{tip_y+r*0.2:.1f}" '
              f'fill="{stroke}" opacity="{opacity}"/>\n')
    return arc + ah


In [ ]:
def _action_glyph(instr: Instruction, cx, cy, size, fill, stroke) -> str:
    """Return SVG for a non-movement instruction. Pure geometry, no text."""
    if instr == Instruction.PAUSE:
        return "".join(svg_pause(cx, cy, size, fill, stroke))
    elif instr == Instruction.DEFEND:
        return svg_shield(cx, cy, size, fill, stroke)
    elif instr == Instruction.HARVEST:
        return svg_sheaf(cx, cy, size, fill, stroke)
    elif instr == Instruction.GIVE:
        return svg_gift(cx, cy, size, fill, stroke)
    elif instr == Instruction.SETTLE:
        return svg_house(cx, cy, size, fill, stroke)
    elif instr == Instruction.ROT_L:
        return svg_rot_arrow(cx, cy, size * 0.45, clockwise=False, stroke=stroke)
    elif instr == Instruction.ROT_R:
        return svg_rot_arrow(cx, cy, size * 0.45, clockwise=True, stroke=stroke)
    return ""


def piece_plan_overlay(steps: list[PieceStep], grid: HexGrid,
                       piece: Piece = None, c2f: dict = None,
                       piece_scale: float = None) -> str:
    """Render movement plan — pure SVG geometry, no emoji or text glyphs."""
    svg, N = "", len(grid.hexes)
    flag = piece.flag if piece else None
    r = grid.radius

    if piece_scale is None:
        piece_scale = r / 45.0

    color = flag.primary if flag else "#4CAF50"
    comp  = flag.comp    if flag else "#333"

    # Heraldic fill for chess pieces
    pat_fill = color
    if flag:
        pat_name = f"piece_pat_{piece.id[:8]}"
        pat = flag.flagPattern(pat_name, scale=0.1)
        grid.builder.add_definition(pat)
        pat_fill = f"url(#{pat_name})"

    glyph_size = max(6, r * 0.45)
    label_size = max(5, r * 0.25)
    dot_r      = max(2, r * 0.12)
    block_r    = max(3, r * 0.18)

    arrow_id = f"plan_arrow_{id(piece) % 9999}"
    arrow_style = StyleCSS(arrow_id,
                           stroke=color, stroke_width=max(1, r * 0.08),
                           fill="none", opacity="0.7",
                           stroke_dasharray=f"{r*0.25:.0f},{r*0.12:.0f}")
    grid.builder.add_style(arrow_style)

    # Start piece
    start_fine = _map_point(piece.location, c2f) if piece and piece.location is not None else -1
    if 0 <= start_fine < N and flag:
        center = grid.hexes[start_fine].center
        renderer = _PIECE_RENDERERS.get(piece.piece_type, 'pawnPiece')
        svg += getattr(flag, renderer)(MapCord(center.x, center.y),
                                       scale=piece_scale,
                                       piece_id=f"plan_start_{piece.id[:8]}",
                                       fill=pat_fill)
        # Start ring marker
        svg += svg_ring(center.x, center.y, r * 0.75, stroke=color, stroke_width=max(1.5, r*0.07))

    for step in steps:
        fine_idx  = _map_point(step.hex_idx, c2f)
        fine_prev = _map_point(step.prev_idx, c2f) if step.prev_idx >= 0 else -1
        if fine_idx < 0 or fine_idx >= N:
            continue
        cx, cy = grid.hexes[fine_idx].center.x, grid.hexes[fine_idx].center.y

        if (step.instruction == Instruction.FORWARD and not step.blocked
                and 0 <= fine_prev < N and fine_prev != fine_idx):
            svg += grid.arrow(fine_prev, fine_idx, style=arrow_style, factor=0.3)
            svg += svg_circle(cx, cy, dot_r, fill=color, stroke="none", opacity=0.55)

        elif step.blocked:
            # Red X using two crossed lines inside a circle
            svg += svg_circle(cx, cy, block_r, fill="red", stroke="darkred",
                              stroke_width=1, opacity=0.55)
            d = block_r * 0.55
            for x1,y1,x2,y2 in [(cx-d,cy-d,cx+d,cy+d),(cx+d,cy-d,cx-d,cy+d)]:
                svg += (f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" '
                        f'stroke="white" stroke-width="{max(1,block_r*0.35):.1f}" '
                        f'stroke-linecap="round"/>\n')

        else:
            svg += _action_glyph(step.instruction, cx, cy, glyph_size, color, comp)

        # Step number — only safe ASCII digits, no special chars
        off = r * 0.3
        svg += (f'<text x="{cx+off:.1f}" y="{cy-off:.1f}" text-anchor="start" '
                f'font-size="{label_size:.0f}" fill="{comp}" '
                f'font-family="sans-serif" font-weight="bold">{step.step+1}</text>\n')

    # Ghost piece at end
    if steps and flag:
        last = steps[-1]
        final_fine = _map_point(last.hex_idx, c2f)
        if 0 <= final_fine < N and final_fine != start_fine:
            end_center = grid.hexes[final_fine].center
            renderer = _PIECE_RENDERERS.get(piece.piece_type, 'pawnPiece')
            svg += getattr(flag, renderer)(MapCord(end_center.x, end_center.y),
                                           scale=piece_scale * 0.7,
                                           piece_id=f"plan_end_{piece.id[:8]}",
                                           fill=pat_fill,
                                           attrs={"opacity": "0.4"})

    return svg


In [ ]:
def piece_plan_overlay(steps: list[PieceStep], grid: HexGrid,
                       piece: Piece = None, c2f: dict = None,
                       piece_scale: float = None) -> str:
    """Render movement plan — pure SVG geometry, no emoji or text glyphs."""
    svg, N = "", len(grid.hexes)
    flag = piece.flag if piece else None
    r = grid.radius

    # Fill ~75% of hex diameter
    if piece_scale is None:
        piece_scale = (r * 0.75) / 22.5  # chess SVGs are centered on 22.5px

    color = flag.primary if flag else "#4CAF50"
    comp  = flag.comp    if flag else "#333"

    # Heraldic fill pattern for chess pieces
    pat_fill = color
    if flag:
        pat_name = f"piece_pat_{piece.id[:8]}"
        pat = flag.flagPattern(pat_name, scale=0.1)
        grid.builder.add_definition(pat)
        pat_fill = f"url(#{pat_name})"

    glyph_size = r * 0.75            # fills 75% of hex
    label_size = max(12, r * 0.3)
    dot_r      = max(2, r * 0.15)
    block_r    = max(4, r * 0.25)

    arrow_id = f"plan_arrow_{id(piece) % 9999}"
    arrow_style = StyleCSS(arrow_id,
                           stroke=color, stroke_width=max(1, r * 0.08),
                           fill="none", opacity="0.7",
                           stroke_dasharray=f"{r*0.25:.0f},{r*0.12:.0f}")
    grid.builder.add_style(arrow_style)

    # Start piece + ring marker
    start_fine = _map_point(piece.location, c2f) if piece and piece.location is not None else -1
    if 0 <= start_fine < N and flag:
        center = grid.hexes[start_fine].center
        renderer = _PIECE_RENDERERS.get(piece.piece_type, 'pawnPiece')
        svg += getattr(flag, renderer)(MapCord(center.x, center.y),
                                       scale=piece_scale,
                                       piece_id=f"plan_start_{piece.id[:8]}",
                                       fill=pat_fill)
        svg += svg_ring(center.x, center.y, r * 0.8, stroke=color,
                        stroke_width=max(1.5, r * 0.07))

    for step in steps:
        fine_idx  = _map_point(step.hex_idx, c2f)
        fine_prev = _map_point(step.prev_idx, c2f) if step.prev_idx >= 0 else -1
        if fine_idx < 0 or fine_idx >= N:
            continue
        cx, cy = grid.hexes[fine_idx].center.x, grid.hexes[fine_idx].center.y

        # === FORWARD → spline arrow + waypoint dot ===
        if (step.instruction == Instruction.FORWARD and not step.blocked
                and 0 <= fine_prev < N and fine_prev != fine_idx):
            svg += grid.arrow(fine_prev, fine_idx, style=arrow_style, factor=0.3)
            svg += svg_circle(cx, cy, dot_r, fill=color, stroke="none", opacity=0.55)

        # === Blocked → red circle with white X ===
        elif step.blocked:
            svg += svg_circle(cx, cy, block_r, fill="red", stroke="darkred",
                              stroke_width=1, opacity=0.55)
            d = block_r * 0.55
            for x1,y1,x2,y2 in [(cx-d,cy-d,cx+d,cy+d),(cx+d,cy-d,cx-d,cy+d)]:
                svg += (f'<line x1="{x1:.1f}" y1="{y1:.1f}" '
                        f'x2="{x2:.1f}" y2="{y2:.1f}" '
                        f'stroke="white" stroke-width="{max(1, block_r*0.35):.1f}" '
                        f'stroke-linecap="round"/>\n')

        # === Action glyph — pure geometry ===
        else:
            svg += _action_glyph(step.instruction, cx, cy, glyph_size, color, comp)

        # Step number — safe ASCII only
        off = r * 0.35
        svg += (f'<text x="{cx+off:.1f}" y="{cy-off:.1f}" text-anchor="start" '
                f'font-size="{label_size:.0f}" fill="{comp}" '
                f'font-family="sans-serif" font-weight="bold">{step.step+1}</text>\n')

    # Ghost piece at final position
    if steps and flag:
        last = steps[-1]
        final_fine = _map_point(last.hex_idx, c2f)
        if 0 <= final_fine < N and final_fine != start_fine:
            end_center = grid.hexes[final_fine].center
            renderer = _PIECE_RENDERERS.get(piece.piece_type, 'pawnPiece')
            svg += getattr(flag, renderer)(MapCord(end_center.x, end_center.y),
                                           scale=piece_scale * 0.7,
                                           piece_id=f"plan_end_{piece.id[:8]}",
                                           fill=pat_fill,
                                           attrs={"opacity": "0.4"})
    return svg


In [ ]:
uid = ensure_user(dummySession)
active = globalStore.active_board(uid)
board, terrain, grid = active.board, active.board.terrain, active.board.terrain.hexGrid

# Find the existing piece (the knight)
kingdom = next(k for k in board.kingdoms
               if any(c for s in k.settlements for c in s.citizens))
piece = next(c for s in kingdom.settlements for c in s.citizens)

# Assign a patrol route: forward-forward-turn, with a harvest stop
piece.rules = [
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.HARVEST.value,
    Instruction.FORWARD.value,
    Instruction.ROT_L.value,
]
piece.patrol = True

# Simulate 20 steps on the coarse grid (cheap — just integer lookups)
steps, region = piece.plan_region(
    grid, terrain.elevations,
    num_rounds=20, padding=4,
    countries=terrain.fields.get("country")
)

print(f"Piece: {piece.piece_type.name} at hex {piece.location}, facing={piece.facing}")
print(f"{len(steps)} steps, {len(region)} hex region\n")
for s in steps:
    tag = " ✘" if s.blocked else ""
    print(f"  {s.step+1:2d}. {s.instruction.name:8s} → hex {s.hex_idx}  f={s.facing}{tag}")

# Zoom into the plan region (chunk cache likely warm)
result = active.cover.zoom_region_fast(region, compute_weather=True)
zoomed = result.terrain
zoomed.hexGrid.adjustRadius(25)
zoomed.colorMap()
zoomed.hexGrid.update()

# Draw the plan overlay on the zoomed grid
overlay = piece_plan_overlay(steps, zoomed.hexGrid,
                             piece=piece, 
                             c2f=result.c2f)

zoomed.hexGrid.builder.adjust("plan", overlay)

#from IPython.display import SVG, display
#display(SVG(zoomed.hexGrid.builder.xml()))
webMe(Div(NotStr(zoomed.hexGrid.builder.xml())))



This is such a strong start. I realizde that it is ambigous as to whether the path is meant for the fine or coarse map. I kind of want to be able to do both. but that might make things too complicated

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

It's actually not that complicated because the overlay function already handles both cases through the `c2f` parameter:

- `c2f=None` → works directly on whichever grid you pass in (coarse)
- `c2f=result.c2f` → maps coarse step indices onto the fine grid

So the same `steps` list (always simulated on coarse) can be rendered on either grid just by changing what you pass. The question is really about **when you'd want each**:

- **Coarse map** (kingdom/world view) — show all pieces' paths at a glance, small scale
- **Fine map** (piece/settlement view) — show one piece's path in detail on zoomed terrain

Does that match your mental model, or are you thinking of cases where that breaks down?

But what if I want to give instructions in terms of fine

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

That's a genuinely different design — right now `Instruction.FORWARD` moves one **coarse hex** per step, but on the fine grid that's `scale²` hexes worth of distance. So if you want fine-grained instructions you'd need to either:

1. **Simulate on the fine grid** — run `simulate` against the zoomed `terrain.hexGrid` + `elevations`, so each `FORWARD` is one fine hex
2. **Scale the move strength** — keep coarse simulation but treat each `FORWARD` as `move_strength` fine hexes (the `move_strength` field on `Piece` already exists for exactly this)

Option 2 is interesting because `KNIGHT` has `move_strength=7`, which would naturally cover more fine hexes per instruction than a `PAWN` with `move_strength=3`.

Which feels more right to you — should a single `FORWARD` instruction always mean "one hex step", or should it mean "move as far as your strength allows"?

I think the quesion is how much automty do I want on the moves. If we keep things a the coarse level (and we will have to balance so that things don't fly around the map) then this would be interpreated as broad instuctions, but then when we resolve all of this it should be at the fine level.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

That's a really clean separation of concerns:

- **Coarse = intent/orders** — broad strategic instructions ("patrol this area", "harvest here"), where `move_strength` acts as a throttle to stop fast pieces dominating
- **Fine = resolution** — when turns actually resolve, the piece's coarse path gets projected to fine hexes and that's where collisions, harvests, and combat are actually computed

So the `PieceStep` list stays coarse for planning/display, but when you "tick" the game you'd walk each step through `c2f` to get the actual fine hexes involved in resolution.

Does that mean `move_strength` should cap how many `FORWARD` steps a piece can take **per turn** rather than per instruction?

Yes move_strength is how many forward steps per turn rather than per instruction. this way uphill which counts twice can be simulated

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

That's a classic hex-game movement budget — each `FORWARD` costs 1 movement point normally, 2 if moving uphill, and the piece stops when its `move_strength` budget is exhausted for that turn.

So `simulate` would need two small changes:

1. Track a `movement_budget` that resets to `move_strength` each turn
2. Deduct 1 (flat/downhill) or 2 (uphill) per `FORWARD`, and skip further `FORWARD` steps that turn if budget is 0

The tricky question: what counts as "one turn" in the rules list? Is it a fixed number of instructions, or does the turn end when the movement budget runs out?

So lets do an example lets say we have a pawn with move strength 3 and rules f f f f f f  (assuming flag) that would be completed in two turns. if a knight have move strength 6 then it would be 1 turn. We can think of move_strength as speed (which might be a better label to refactor), but the speed of how fast you go through the rules

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠

Exactly — so `move_strength` (or `speed`) is the number of `FORWARD` instructions consumed per turn, and the rules list just loops. So:

- **Pawn** (speed=3): `F F F | F F F` — turn 1 uses steps 1-3, turn 2 uses steps 4-6
- **Knight** (speed=6): `F F F F F F` — done in 1 turn

And uphill would cost 2 against the budget, so a pawn hitting a hill mid-turn might only get 1 more step that turn instead of 2.

That means `simulate` should track `(turn, budget_remaining)` per step, so you can later ask "what did this piece do in turn N?" — useful for both the overlay and resolution.

Does a `ROT_L/ROT_R` cost movement budget, or is rotation free?

I am going to work through some thoughts

## Modified Gosper
so the rules on Piece is going to be reworked as a list of ints with the following
0 -> pause
1 -> defend # in defend mode we get more aggressive
2 -> harvest # gather food from the hex
3 -> give   # give food to either a neighbor piece or a settlement
4 -> settle or siege # reserved for figuring out how to conquer settlements
5 -> forward # move on place forward (depending on movement cost). If you can't move forward (ie you hit water) rotate left. if it is there is another piece in the way pause   
6 -> rotate left
7 -> rotate right


## Movement costs

we have a movement cost formula
```python
@patch
def movement_cost(self: GameBoard, from_pos: HexPosition, to_pos: HexPosition, origin: int) -> float:
    """Calculate cost to move from one hex to another using HexPositions.
    
    Args:
        from_pos: Starting HexPosition
        to_pos: Destination HexPosition
        origin: Origin index for converting HexPosition to grid index
    
    Returns infinity for invalid moves (water, out of bounds).
    """
    terrain = self.terrain
    grid = terrain.hexGrid
    
    # Convert to indices to check terrain
    to_idx = grid.hexposition_to_index(to_pos, origin)
    from_idx = grid.hexposition_to_index(from_pos, origin)
    
    # Check bounds
    if to_idx < 0 or to_idx >= len(terrain.elevations):
        return float('inf')
    
    # Can't cross water
    if terrain.elevations[to_idx] < 1:
        return float('inf')
    
    from_elev = terrain.elevations[from_idx]
    to_elev = terrain.elevations[to_idx]
    
    # Base cost
    base_cost = 1.0
    
    # Elevation gain penalty (climbing is expensive)
    elev_diff = to_elev - from_elev
    if elev_diff > 0:
        # Exponential penalty for climbing
        base_cost += elev_diff * 2.0
    
    return base_cost
```
Which we will need to figure out how to use for a turn. movement_range will tell how many moves we can do. Lets make turns free (including turns from hitting the ocean). the first 4 counts as a single movement cost.

## food and diet
we are going to add three fields 
1. which is the amount of food a piece has
2. its diet. how much does it eat per turn If it rounds out of food it will start to lose health. 
3. food capacity which is how much total food it can carry.

## deprecate
we don't need memory since that will be in rules.

## settle
We are going to keep these around even though I haven't figure them out
harvest_goal: Resources = None
settle_progress: int = 0
settle_threshold: int = 3
sight = 3

## The things we carry
Lets start thinking about the rules editor for a piece
1. we are going to want a list of rules that are assigned and be able to add and delete rows
2. we are going to want an edit point where we can insert new items
3. we are going to want a HexDragMap that we can add items. The path we make needs to be converted into instructions on how we get there.
4. we need an apply button and the number of rounds. This will be our main map showing where the piece is going to move each turn for the number of turns. I envision this as a series of arrows and circles.
a. arrows are for movement. straight line if single cost squiggle line if multiple cost. the color of the arrow is how aggressive it is.
b. circles are for the pause actions (0 -> pause
1 -> defend # in defend mode we get more aggressive
2 -> harvest # gather food from the hex
3 -> give   # give food to either a neighbor piece or a settlement
) I think red should be for defend.

eventually we are going to want a drop down of a list of common rule sets (patrol around a circle. go out and harvest and comeback, attack by first flanking left). 
we will certainly want to have a hex destination that returns a set of rules on how to get there (hello djisktra)

I need to be clearer on some definitons
1. `round` is for all player and all pieces. Each player gives the set of orders it wants its pieces to do. We need to build out how to carry them out, but I am tempted just to start with the kindergaren model of everybody does what it wants
2. `turn` is equivalent to a round, but just on a single piece. `movement_cost` is how many things it can do in a turn. Some (like rotation) are free. Others like mmoving up a hill might cost a couple of movements.

It can be the case that rule ends and there are spare items. So if the last item in the list is one of the non movement items. ie (like pause) then we just pause the remaining amount of time. if it is a movement (or rotate) then we circle back to the begining.

## Pulling Ranks
so things are prioritzed in chess order ie kings move first and eat first. then queens. This way if a king will move to a spot before a pawn would

I am going to have the flow of the hex be the amount you can harvest. Settlments are going to have build queues. I need to figure out the mechanics of adding settlements whether there are pre existing locations or it behavior on a item

**7. Food economy**

Ill need a `hex_food_yield(terrain, idx) -> float` keyed off climate and flow.



**3. Forward-blocked spin trap**

Instruction 4 says "rotate left if water, pause if piece." But imagine a piece in a cove with water on 5 sides — it rotates 5 times but each time if the chance to move becomes available



Here's where we've landed:

**Budget model** — `move_strength` (rename to `speed`) is the per-turn movement budget, not per-instruction.

| Instruction | Cost | Notes |
|---|---|---|
| `FORWARD` (5) | 1 flat/downhill, 2 uphill | Can't move if budget < cost → turn ends |
| `ROT_L` (6) / `ROT_R` (7) | free | Including auto-rotations from hitting water |
| `PAUSE` (0) | 1 | Explicit wait |
| `DEFEND` (1) | 1 | Boosts combat response |
| `HARVEST` (2) | 1 | Gathers `hex_food_yield` into `food` (capped at `food_capacity`) |
| `GIVE` (3) | 1 | Transfer food to adjacent piece or settlement |
| `SETTLE` (4) | 1 | Reserved — siege/founding |

**FORWARD special cases:**
- **Water/edge** → auto-rotate left (free), up to 5×. Full circle → stuck, forced PAUSE (costs 1)
- **Occupied hex** → forced PAUSE (costs 1), cursor still advances

**End-of-rules:**
- Last instruction is non-movement (`PAUSE`/`DEFEND`/`HARVEST`/`GIVE`/`SETTLE`) → idle out remaining budget
- Last instruction is movement/rotation → wrap cursor to 0, keep going

**Turn resolution order:**
- KING → QUEEN → BISHOP → KNIGHT → ROOK → PAWN (chess rank priority)

**Food economy:**
- `food` — current supply
- `diet` — consumed per turn (deducted at turn end)
- `food_capacity` — max carried
- `food < 0` → lose health

**Example: Pawn (speed=3), rules `F F F F F F`, flat terrain:**
```
Turn 1: F(1) F(1) F(1) → budget 0
Turn 2: F(1) F(1) F(1) → budget 0
```

**Example: Pawn (speed=3), rules `F F F F F F`, uphill at step 2:**
```
Turn 1: F(1) F_uphill(2) → budget 0 (only 2 hexes moved)
Turn 2: F(1) F(1) F(1)  → budget 0
```

**Example: Pawn (speed=3), rules `F F HARVEST`:**
```
Turn 1: F(1) F(1) HARVEST(1) → end of rules, last=HARVEST → idle
Turn 2: same pattern (wraps because patrol=True)
```



yes on keeping track on siumale. I realize you didn't have this design above. Can you flesh it out more now?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's the full revised simulation with turn budgets, movement costs, auto-rotation, and end-of-rules behavior:

```python
@dataclass
class PieceStep:
    """One resolved step in a piece's movement plan."""
    hex_idx: int
    facing: int
    instruction: Instruction
    prev_idx: int
    step: int               # global step counter (across all turns)
    turn: int               # which turn (0-indexed)
    cost: float             # movement points consumed by this step
    budget_remaining: float # budget left after this step
    blocked: bool = False
    auto_rotations: int = 0 # free rotations from hitting water/edge

# Cost categories
_COSTS_ONE = {Instruction.PAUSE, Instruction.DEFEND, Instruction.HARVEST,
              Instruction.GIVE, Instruction.SETTLE}
_FREE = {Instruction.ROT_L, Instruction.ROT_R}

def _move_cost(elevations, from_idx, to_idx) -> float:
    """Simple movement cost: 1 flat/downhill, 2 uphill."""
    diff = elevations[to_idx] - elevations[from_idx]
    return 2.0 if diff > 0 else 1.0

@patch
def simulate(self: Piece, grid: HexGrid, elevations: np.ndarray,
             num_turns: int = 10,
             countries: np.ndarray = None,
             occupied: set[int] = None) -> list['PieceStep']:
    """Simulate piece over multiple turns using move_strength as per-turn budget.

    Budget rules:
      - FORWARD: costs _move_cost (1 flat, 2 uphill). If can't afford → end turn.
      - ROT_L / ROT_R: free
      - PAUSE / DEFEND / HARVEST / GIVE / SETTLE: cost 1
      - FORWARD into water/edge: auto-rotate left (free, up to 5×). Full circle → stuck, cost 1.
      - FORWARD into occupied hex: pause instead, cost 1.

    End-of-rules:
      - Last instruction was non-movement → pause out remaining budget.
      - Last instruction was movement/rotate → wrap cursor to 0.
    """
    if not self.rules or self.location is None:
        return []

    occupied = set(occupied) if occupied else set()
    steps = []
    pos, fac, cur = self.location, self.facing, self.cursor
    step_n = 0

    for turn in range(num_turns):
        budget = float(self.move_strength)

        while budget > 0:
            # --- End of rules ---
            if cur >= len(self.rules):
                last = Instruction(self.rules[-1])
                if last in _COSTS_ONE:
                    # Idle out the turn
                    steps.append(PieceStep(pos, fac, Instruction.PAUSE,
                                           pos, step_n, turn, budget, 0.0))
                    step_n += 1
                    break
                cur = 0  # wrap around

            instr = Instruction(self.rules[cur])
            prev = pos

            # --- FREE: rotations ---
            if instr in _FREE:
                fac = ((fac - 1) if instr == Instruction.ROT_L
                       else (fac + 1)) % 6
                steps.append(PieceStep(pos, fac, instr, pos,
                                       step_n, turn, 0.0, budget))
                step_n += 1; cur += 1
                continue

            # --- COST 1: non-movement actions ---
            if instr in _COSTS_ONE:
                budget -= 1.0
                steps.append(PieceStep(pos, fac, instr, pos,
                                       step_n, turn, 1.0, budget))
                step_n += 1; cur += 1
                continue

            # --- FORWARD ---
            auto_rots = 0
            moved = False
            saved_fac = fac

            while auto_rots <= 5:
                d = HexPosition.directions()[fac % 6]
                nbr = grid.hexposition_to_index(d, pos)

                passable = (0 <= nbr < len(elevations)
                            and nbr not in grid.invalidRegion
                            and elevations[nbr] >= 1)
                if passable and countries is not None:
                    passable = countries[nbr] >= 0

                if not passable:
                    fac = (fac - 1) % 6   # free auto-rotate left
                    auto_rots += 1
                    continue

                # Occupied → forced pause
                if nbr in occupied:
                    steps.append(PieceStep(pos, fac, Instruction.PAUSE, pos,
                                           step_n, turn, 1.0, budget - 1.0,
                                           blocked=True,
                                           auto_rotations=auto_rots))
                    budget -= 1.0
                    step_n += 1; cur += 1
                    moved = True
                    break

                # Can we afford it?
                cost = _move_cost(elevations, pos, nbr)
                if cost > budget:
                    budget = 0       # turn over, can't afford
                    moved = True
                    break

                # Move
                occupied.discard(pos)
                pos = nbr
                occupied.add(pos)
                budget -= cost
                steps.append(PieceStep(pos, fac, Instruction.FORWARD, prev,
                                       step_n, turn, cost, budget,
                                       auto_rotations=auto_rots))
                step_n += 1; cur += 1
                moved = True
                break

            if not moved:
                # Spun full circle — stuck
                fac = saved_fac      # 6 left rotations = back to original
                steps.append(PieceStep(pos, fac, Instruction.PAUSE, pos,
                                       step_n, turn, 1.0, budget - 1.0,
                                       blocked=True, auto_rotations=6))
                budget -= 1.0
                step_n += 1; cur += 1

    return steps


def steps_by_turn(steps: list[PieceStep]) -> dict[int, list[PieceStep]]:
    """Group steps by turn number."""
    turns = {}
    for s in steps:
        turns.setdefault(s.turn, []).append(s)
    return turns


@patch
def plan_region(self: Piece, grid: HexGrid, elevations: np.ndarray,
                num_turns: int = 10, padding: int = 3,
                countries: np.ndarray = None,
                occupied: set[int] = None
                ) -> tuple[list['PieceStep'], 'HexRegion']:
    """Simulate and return (steps, bounding_region) for zooming."""
    steps = self.simulate(grid, elevations, num_turns,
                          countries=countries, occupied=occupied)
    visited = {self.location}
    for s in steps:
        visited.add(s.hex_idx)
    expanded = set()
    for h in visited:
        for n in grid.indices_in_range(h, padding):
            if 0 <= n < len(elevations):
                expanded.add(n)
    return steps, HexRegion(hexes=expanded, hexGrid=grid)
```

Key behaviors to verify:

| Scenario | What happens |
|---|---|
| Pawn (speed 3), rules `F F F F F F` | Turn 1: 3 flat moves. Turn 2: 3 more. |
| Knight (speed 7), same rules | Turn 1: 6 moves + wraps, 1 more from start. |
| Pawn hits uphill on step 2 | Step 1 costs 1, step 2 costs 2 → budget 0, turn over after 2 steps. |
| Forward into water | Auto-rotates left (free) until land found, then moves. |
| Trapped in cove (water on all 6 sides) | Spins 6×, gives up → forced PAUSE costs 1. |
| Rules = `F F HARVEST`, speed=5 | F(1) F(1) HARVEST(1) → cursor at end, last is HARVEST (non-movement) → pause remaining 2 budget. |
| Rules = `F F ROT_R`, speed=5 | F(1) F(1) ROT_R(free) → cursor at end, last is ROT_R (movement-like) → wraps to 0 → F(1) F(1) ROT_R(free) → budget=1 → wraps → F(1) done. |
| Forward into occupied hex | Forced PAUSE, costs 1, still advances cursor. |

## Revised Movement

In [ ]:
@dataclass
class PieceStep:
    """One resolved step in a piece's movement plan."""
    hex_idx: int
    facing: int
    instruction: Instruction
    prev_idx: int
    step: int               # global step counter (across all turns)
    turn: int               # which turn (0-indexed)
    cost: float             # movement points consumed by this step
    budget_remaining: float # budget left after this step
    blocked: bool = False
    auto_rotations: int = 0 # free rotations from hitting water/edge

# Cost categories
_COSTS_ONE = {Instruction.PAUSE, Instruction.DEFEND, Instruction.HARVEST,
              Instruction.GIVE, Instruction.SETTLE}
_FREE = {Instruction.ROT_L, Instruction.ROT_R}

def _move_cost(elevations, from_idx, to_idx) -> float:
    """Simple movement cost: 1 flat/downhill, 2 uphill."""
    diff = elevations[to_idx] - elevations[from_idx]
    return 2.0 if diff > 0 else 1.0

@patch
def simulate(self: Piece, grid: HexGrid, elevations: np.ndarray,
             num_turns: int = 10,
             countries: np.ndarray = None,
             occupied: set[int] = None) -> list['PieceStep']:
    """Simulate piece over multiple turns using move_strength as per-turn budget.

    Budget rules:
      - FORWARD: costs _move_cost (1 flat, 2 uphill). If can't afford → end turn.
      - ROT_L / ROT_R: free
      - PAUSE / DEFEND / HARVEST / GIVE / SETTLE: cost 1
      - FORWARD into water/edge: auto-rotate left (free, up to 5×). Full circle → stuck, cost 1.
      - FORWARD into occupied hex: pause instead, cost 1.

    End-of-rules:
      - Last instruction was non-movement → pause out remaining budget.
      - Last instruction was movement/rotate → wrap cursor to 0.
    """
    if not self.rules or self.location is None:
        return []

    occupied = set(occupied) if occupied else set()
    steps = []
    pos, fac, cur = self.location, self.facing, self.cursor
    step_n = 0

    for turn in range(num_turns):
        budget = float(self.move_strength)

        while budget > 0:
            # --- End of rules ---
            if cur >= len(self.rules):
                last = Instruction(self.rules[-1])
                if last in _COSTS_ONE:
                    # Idle out the turn
                    steps.append(PieceStep(pos, fac, Instruction.PAUSE,
                                           pos, step_n, turn, budget, 0.0))
                    step_n += 1
                    break
                cur = 0  # wrap around

            instr = Instruction(self.rules[cur])
            prev = pos

            # --- FREE: rotations ---
            if instr in _FREE:
                fac = ((fac - 1) if instr == Instruction.ROT_L
                       else (fac + 1)) % 6
                steps.append(PieceStep(pos, fac, instr, pos,
                                       step_n, turn, 0.0, budget))
                step_n += 1; cur += 1
                continue

            # --- COST 1: non-movement actions ---
            if instr in _COSTS_ONE:
                budget -= 1.0
                steps.append(PieceStep(pos, fac, instr, pos,
                                       step_n, turn, 1.0, budget))
                step_n += 1; cur += 1
                continue

            # --- FORWARD ---
            auto_rots = 0
            moved = False
            saved_fac = fac

            while auto_rots <= 5:
                d = HexPosition.directions()[fac % 6]
                nbr = grid.hexposition_to_index(d, pos)

                passable = (0 <= nbr < len(elevations)
                            and nbr not in grid.invalidRegion
                            and elevations[nbr] >= 1)
                if passable and countries is not None:
                    passable = countries[nbr] >= 0

                if not passable:
                    fac = (fac - 1) % 6   # free auto-rotate left
                    auto_rots += 1
                    continue

                # Occupied → forced pause
                if nbr in occupied:
                    steps.append(PieceStep(pos, fac, Instruction.PAUSE, pos,
                                           step_n, turn, 1.0, budget - 1.0,
                                           blocked=True,
                                           auto_rotations=auto_rots))
                    budget -= 1.0
                    step_n += 1; cur += 1
                    moved = True
                    break

                # Can we afford it?
                cost = _move_cost(elevations, pos, nbr)
                if cost > budget:
                    budget = 0       # turn over, can't afford
                    moved = True
                    break

                # Move
                occupied.discard(pos)
                pos = nbr
                occupied.add(pos)
                budget -= cost
                steps.append(PieceStep(pos, fac, Instruction.FORWARD, prev,
                                       step_n, turn, cost, budget,
                                       auto_rotations=auto_rots))
                step_n += 1; cur += 1
                moved = True
                break

            if not moved:
                # Spun full circle — stuck
                fac = saved_fac      # 6 left rotations = back to original
                steps.append(PieceStep(pos, fac, Instruction.PAUSE, pos,
                                       step_n, turn, 1.0, budget - 1.0,
                                       blocked=True, auto_rotations=6))
                budget -= 1.0
                step_n += 1; cur += 1

    return steps


def steps_by_turn(steps: list[PieceStep]) -> dict[int, list[PieceStep]]:
    """Group steps by turn number."""
    turns = {}
    for s in steps:
        turns.setdefault(s.turn, []).append(s)
    return turns


@patch
def plan_region(self: Piece, grid: HexGrid, elevations: np.ndarray,
                num_turns: int = 10, padding: int = 3,
                countries: np.ndarray = None,
                occupied: set[int] = None
                ) -> tuple[list['PieceStep'], 'HexRegion']:
    """Simulate and return (steps, bounding_region) for zooming."""
    steps = self.simulate(grid, elevations, num_turns,
                          countries=countries, occupied=occupied)
    visited = {self.location}
    for s in steps:
        visited.add(s.hex_idx)
    expanded = set()
    for h in visited:
        for n in grid.indices_in_range(h, padding):
            if 0 <= n < len(elevations):
                expanded.add(n)
    return steps, HexRegion(hexes=expanded, hexGrid=grid)


Can we do a simulation to test to see if this works at all?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
uid = ensure_user(dummySession)
active = globalStore.active_board(uid)
board, terrain, grid = active.board, active.board.terrain, active.board.terrain.hexGrid

# Find the knight
kingdom = next(k for k in board.kingdoms
               if any(c for s in k.settlements for c in s.citizens))
piece = next(c for s in kingdom.settlements for c in s.citizens)

# Set up rules: forward-forward-rotate-forward-forward-harvest (patrol + forage)
piece.rules = [
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.HARVEST.value,
]
piece.patrol = True
piece.facing = 0

print(f"Piece: {piece.piece_type.name}, speed={piece.move_strength}, loc={piece.location}")
print(f"Rules: {[Instruction(r).name for r in piece.rules]}")
print(f"Patrol: {piece.patrol}\n")

# Simulate 5 turns
steps = piece.simulate(grid, terrain.elevations, num_turns=5,
                       countries=terrain.fields.get("country"))

# Print turn-by-turn
by_turn = steps_by_turn(steps)
for t in sorted(by_turn.keys()):
    print(f"── Turn {t+1} ──")
    for s in by_turn[t]:
        moved = f"→ hex {s.hex_idx}" if s.hex_idx != s.prev_idx else f"@ hex {s.hex_idx}"
        rot = f" (auto-rot {s.auto_rotations}×)" if s.auto_rotations else ""
        blk = " BLOCKED" if s.blocked else ""
        print(f"  step {s.step:2d}: {s.instruction.name:8s} {moved}  "
              f"face={s.facing}  cost={s.cost:.0f}  budget={s.budget_remaining:.0f}{rot}{blk}")
    print()

print(f"Total steps: {len(steps)}")
print(f"Final position: hex {steps[-1].hex_idx}, facing {steps[-1].facing}")
```

This will show us:
- Whether the budget is being consumed correctly (knight speed=7 should eat through more rules per turn)
- Whether rotations are free
- Whether HARVEST costs 1
- Whether wrapping at end-of-rules works (patrol mode)
- Whether uphill costs 2 vs flat costs 1

Run it and let's see what comes out!

In [ ]:
uid = ensure_user(dummySession)
active = globalStore.active_board(uid)
board, terrain, grid = active.board, active.board.terrain, active.board.terrain.hexGrid

# Find the knight
kingdom = next(k for k in board.kingdoms
               if any(c for s in k.settlements for c in s.citizens))
piece = next(c for s in kingdom.settlements for c in s.citizens)

# Set up rules: forward-forward-rotate-forward-forward-harvest (patrol + forage)
piece.rules = [
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.HARVEST.value,
]
piece.patrol = True
piece.facing = 0

print(f"Piece: {piece.piece_type.name}, speed={piece.move_strength}, loc={piece.location}")
print(f"Rules: {[Instruction(r).name for r in piece.rules]}")
print(f"Patrol: {piece.patrol}\n")

# Simulate 5 turns
steps = piece.simulate(grid, terrain.elevations, num_turns=5,
                       countries=terrain.fields.get("country"))

# Print turn-by-turn
by_turn = steps_by_turn(steps)
for t in sorted(by_turn.keys()):
    print(f"── Turn {t+1} ──")
    for s in by_turn[t]:
        moved = f"→ hex {s.hex_idx}" if s.hex_idx != s.prev_idx else f"@ hex {s.hex_idx}"
        rot = f" (auto-rot {s.auto_rotations}×)" if s.auto_rotations else ""
        blk = " BLOCKED" if s.blocked else ""
        print(f"  step {s.step:2d}: {s.instruction.name:8s} {moved}  "
              f"face={s.facing}  cost={s.cost:.0f}  budget={s.budget_remaining:.0f}{rot}{blk}")
    print()

print(f"Total steps: {len(steps)}")
print(f"Final position: hex {steps[-1].hex_idx}, facing {steps[-1].facing}")


What do you think. Should we hook it up to an overlay demo

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The simulation is working — but look at turns 3–5: the piece is stuck idling forever. That's because `HARVEST` is the last rule, so at end-of-rules it idles the remaining budget... and `cur` never resets for the next turn. Quick fix — reset `cur=0` when patrol is true at turn start:

```python
    for turn in range(num_turns):
        budget = float(self.move_strength)
        # Reset cursor at turn start if patrol and we've exhausted rules
        if self.patrol and cur >= len(self.rules):
            cur = 0
```

Add that one `if` right after `budget = float(...)` in `simulate`, then let's see the overlay:

```python
# Re-simulate with the fix applied
steps, region = piece.plan_region(grid, terrain.elevations,
                                  num_turns=5, padding=4,
                                  countries=terrain.fields.get("country"))

print(f"{len(steps)} steps, {len(region)} hex region")

# Zoom
result = active.cover.zoom_region_fast(region, compute_weather=True)
zoomed = result.terrain
zoomed.hexGrid.adjustRadius(25)
zoomed.colorMap()
zoomed.hexGrid.update()

# Draw plan overlay
overlay = piece_plan_overlay(steps, zoomed.hexGrid,
                             piece=piece, c2f=result.c2f)
zoomed.hexGrid.builder.adjust("plan", overlay)

webMe(Div(NotStr(zoomed.hexGrid.builder.xml())))
```

Fix the `simulate` first, then run both cells. You should see the knight's patrol route drawn out across 5 turns with dashed arrows for movement, wheat sheafs for harvests, and a ghost piece at the final position.

In [ ]:
# Re-simulate with the fix applied
steps, region = piece.plan_region(grid, terrain.elevations,
                                  num_turns=5, padding=4,
                                  countries=terrain.fields.get("country"))

print(f"{len(steps)} steps, {len(region)} hex region")

# Zoom
result = active.cover.zoom_region_fast(region, compute_weather=True)
zoomed = result.terrain
zoomed.hexGrid.adjustRadius(25)
zoomed.colorMap()
zoomed.hexGrid.update()

# Draw plan overlay
overlay = piece_plan_overlay(steps, zoomed.hexGrid,
                             piece=piece, c2f=result.c2f)
zoomed.hexGrid.builder.adjust("plan", overlay)

webMe(Div(NotStr(zoomed.hexGrid.builder.xml())))


In [ ]:
#webMe(Div(NotStr(zoomed.hexGrid.builder.xml())))


In [ ]:
!cat ../../HexMagic/terrainpatterns.py

Just to be safe we should not use off menu strings. We have an okay drawing system. We could for instance use circles and draw arcs. If we need a geometric shape lets build it for terrain patterns so we could use it

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good call — emoji in SVG is asking for trouble (XML escaping, font availability, browser differences). Pure SVG geometry is always safe. Here's a set of shape builders that fit the existing style, plus updated overlay glyphs:

**Block 1: SVG geometry helpers (add to a new `game/shapes.py` or alongside the overlay)**

```python
def svg_circle(cx, cy, r, fill="none", stroke="#333", stroke_width=1.5, opacity=1.0, **kw):
    """Filled or outlined circle."""
    return (f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{r:.1f}" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="{stroke_width}" '
            f'opacity="{opacity}"/>\n')

def svg_ring(cx, cy, r, stroke, stroke_width=2, opacity=0.9):
    """Hollow ring (start marker, selection, etc.)."""
    return svg_circle(cx, cy, r, fill="none", stroke=stroke,
                      stroke_width=stroke_width, opacity=opacity)

def svg_arc(cx, cy, r, start_deg, end_deg, stroke, stroke_width=2, opacity=0.8):
    """Arc segment — good for partial rings, rotation indicators."""
    import math
    s = math.radians(start_deg)
    e = math.radians(end_deg)
    x1, y1 = cx + r * math.cos(s), cy + r * math.sin(s)
    x2, y2 = cx + r * math.cos(e), cy + r * math.sin(e)
    large = 1 if (end_deg - start_deg) % 360 > 180 else 0
    return (f'<path d="M {x1:.1f},{y1:.1f} A {r:.1f},{r:.1f} 0 {large} 1 {x2:.1f},{y2:.1f}" '
            f'fill="none" stroke="{stroke}" stroke-width="{stroke_width}" '
            f'opacity="{opacity}" stroke-linecap="round"/>\n')

def svg_shield(cx, cy, size, fill, stroke, opacity=0.85):
    """Shield shape for DEFEND — classic heraldic proportions."""
    w, h = size, size * 1.2
    return (f'<path d="M {cx:.1f},{cy-h*0.5:.1f} '
            f'L {cx+w*0.5:.1f},{cy-h*0.5:.1f} '
            f'L {cx+w*0.5:.1f},{cy+h*0.1:.1f} '
            f'Q {cx:.1f},{cy+h*0.5:.1f} {cx-w*0.5:.1f},{cy+h*0.1:.1f} '
            f'L {cx-w*0.5:.1f},{cy-h*0.5:.1f} Z" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')

def svg_sheaf(cx, cy, size, fill, stroke, opacity=0.85):
    """Wheat sheaf for HARVEST — three lines bound in middle."""
    s = size
    svg = ""
    # Three stalks
    for dx in [-s*0.3, 0, s*0.3]:
        svg += (f'<line x1="{cx+dx:.1f}" y1="{cy-s*0.5:.1f}" '
                f'x2="{cx+dx*0.3:.1f}" y2="{cy+s*0.5:.1f}" '
                f'stroke="{fill}" stroke-width="{max(1.5, s*0.15):.1f}" '
                f'stroke-linecap="round" opacity="{opacity}"/>\n')
    # Binding band
    svg += (f'<line x1="{cx-s*0.35:.1f}" y1="{cy:.1f}" '
            f'x2="{cx+s*0.35:.1f}" y2="{cy:.1f}" '
            f'stroke="{stroke}" stroke-width="{max(1, s*0.1):.1f}" opacity="{opacity}"/>\n')
    return svg

def svg_gift(cx, cy, size, fill, stroke, opacity=0.85):
    """Simple box with ribbon for GIVE."""
    s = size * 0.8
    svg = (f'<rect x="{cx-s*0.5:.1f}" y="{cy-s*0.4:.1f}" '
           f'width="{s:.1f}" height="{s*0.8:.1f}" '
           f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
           f'rx="{s*0.1:.1f}" opacity="{opacity}"/>\n')
    # Ribbon cross
    svg += (f'<line x1="{cx:.1f}" y1="{cy-s*0.4:.1f}" '
            f'x2="{cx:.1f}" y2="{cy+s*0.4:.1f}" '
            f'stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    svg += (f'<line x1="{cx-s*0.5:.1f}" y1="{cy:.1f}" '
            f'x2="{cx+s*0.5:.1f}" y2="{cy:.1f}" '
            f'stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    return svg

def svg_house(cx, cy, size, fill, stroke, opacity=0.85):
    """Simple house for SETTLE."""
    s = size
    # Walls
    svg = (f'<rect x="{cx-s*0.4:.1f}" y="{cy:.1f}" '
           f'width="{s*0.8:.1f}" height="{s*0.5:.1f}" '
           f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    # Roof
    svg += (f'<polygon points="'
            f'{cx-s*0.5:.1f},{cy:.1f} '
            f'{cx:.1f},{cy-s*0.45:.1f} '
            f'{cx+s*0.5:.1f},{cy:.1f}" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    return svg

def svg_pause(cx, cy, size, fill, stroke, opacity=0.7):
    """Two vertical bars for PAUSE."""
    w, h = size * 0.2, size * 0.6
    for dx in [-size*0.18, size*0.18]:
        fill_c = (f'<rect x="{cx+dx-w/2:.1f}" y="{cy-h/2:.1f}" '
                  f'width="{w:.1f}" height="{h:.1f}" '
                  f'fill="{fill}" stroke="{stroke}" stroke-width="1" '
                  f'rx="{w*0.2:.1f}" opacity="{opacity}"/>\n')
        yield fill_c

def svg_rot_arrow(cx, cy, r, clockwise: bool, stroke, stroke_width=2, opacity=0.85):
    """Circular arc arrow for ROT_L / ROT_R."""
    # 270° arc + arrowhead
    if clockwise:
        arc = svg_arc(cx, cy, r, -120, 150, stroke, stroke_width, opacity)
        # Arrowhead tip at end_deg=150
        import math
        tip_x = cx + r * math.cos(math.radians(150))
        tip_y = cy + r * math.sin(math.radians(150))
        ah = (f'<polygon points="'
              f'{tip_x:.1f},{tip_y:.1f} '
              f'{tip_x-r*0.25:.1f},{tip_y-r*0.15:.1f} '
              f'{tip_x-r*0.1:.1f},{tip_y+r*0.2:.1f}" '
              f'fill="{stroke}" opacity="{opacity}"/>\n')
    else:
        arc = svg_arc(cx, cy, r, 30, 300, stroke, stroke_width, opacity)
        import math
        tip_x = cx + r * math.cos(math.radians(30))
        tip_y = cy + r * math.sin(math.radians(30))
        ah = (f'<polygon points="'
              f'{tip_x:.1f},{tip_y:.1f} '
              f'{tip_x+r*0.25:.1f},{tip_y-r*0.15:.1f} '
              f'{tip_x+r*0.1:.1f},{tip_y+r*0.2:.1f}" '
              f'fill="{stroke}" opacity="{opacity}"/>\n')
    return arc + ah
```

**Block 2: Updated overlay — no strings, all geometry**

```python
def _action_glyph(instr: Instruction, cx, cy, size, fill, stroke) -> str:
    """Return SVG for a non-movement instruction. Pure geometry, no text."""
    if instr == Instruction.PAUSE:
        return "".join(svg_pause(cx, cy, size, fill, stroke))
    elif instr == Instruction.DEFEND:
        return svg_shield(cx, cy, size, fill, stroke)
    elif instr == Instruction.HARVEST:
        return svg_sheaf(cx, cy, size, fill, stroke)
    elif instr == Instruction.GIVE:
        return svg_gift(cx, cy, size, fill, stroke)
    elif instr == Instruction.SETTLE:
        return svg_house(cx, cy, size, fill, stroke)
    elif instr == Instruction.ROT_L:
        return svg_rot_arrow(cx, cy, size * 0.45, clockwise=False, stroke=stroke)
    elif instr == Instruction.ROT_R:
        return svg_rot_arrow(cx, cy, size * 0.45, clockwise=True, stroke=stroke)
    return ""


def piece_plan_overlay(steps: list[PieceStep], grid: HexGrid,
                       piece: Piece = None, c2f: dict = None,
                       piece_scale: float = None) -> str:
    """Render movement plan — pure SVG geometry, no emoji or text glyphs."""
    svg, N = "", len(grid.hexes)
    flag = piece.flag if piece else None
    r = grid.radius

    if piece_scale is None:
        piece_scale = r / 45.0

    color = flag.primary if flag else "#4CAF50"
    comp  = flag.comp    if flag else "#333"

    # Heraldic fill for chess pieces
    pat_fill = color
    if flag:
        pat_name = f"piece_pat_{piece.id[:8]}"
        pat = flag.flagPattern(pat_name, scale=0.1)
        grid.builder.add_definition(pat)
        pat_fill = f"url(#{pat_name})"

    glyph_size = max(6, r * 0.45)
    label_size = max(5, r * 0.25)
    dot_r      = max(2, r * 0.12)
    block_r    = max(3, r * 0.18)

    arrow_id = f"plan_arrow_{id(piece) % 9999}"
    arrow_style = StyleCSS(arrow_id,
                           stroke=color, stroke_width=max(1, r * 0.08),
                           fill="none", opacity="0.7",
                           stroke_dasharray=f"{r*0.25:.0f},{r*0.12:.0f}")
    grid.builder.add_style(arrow_style)

    # Start piece
    start_fine = _map_point(piece.location, c2f) if piece and piece.location is not None else -1
    if 0 <= start_fine < N and flag:
        center = grid.hexes[start_fine].center
        renderer = _PIECE_RENDERERS.get(piece.piece_type, 'pawnPiece')
        svg += getattr(flag, renderer)(MapCord(center.x, center.y),
                                       scale=piece_scale,
                                       piece_id=f"plan_start_{piece.id[:8]}",
                                       fill=pat_fill)
        # Start ring marker
        svg += svg_ring(center.x, center.y, r * 0.75, stroke=color, stroke_width=max(1.5, r*0.07))

    for step in steps:
        fine_idx  = _map_point(step.hex_idx, c2f)
        fine_prev = _map_point(step.prev_idx, c2f) if step.prev_idx >= 0 else -1
        if fine_idx < 0 or fine_idx >= N:
            continue
        cx, cy = grid.hexes[fine_idx].center.x, grid.hexes[fine_idx].center.y

        if (step.instruction == Instruction.FORWARD and not step.blocked
                and 0 <= fine_prev < N and fine_prev != fine_idx):
            svg += grid.arrow(fine_prev, fine_idx, style=arrow_style, factor=0.3)
            svg += svg_circle(cx, cy, dot_r, fill=color, stroke="none", opacity=0.55)

        elif step.blocked:
            # Red X using two crossed lines inside a circle
            svg += svg_circle(cx, cy, block_r, fill="red", stroke="darkred",
                              stroke_width=1, opacity=0.55)
            d = block_r * 0.55
            for x1,y1,x2,y2 in [(cx-d,cy-d,cx+d,cy+d),(cx+d,cy-d,cx-d,cy+d)]:
                svg += (f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" '
                        f'stroke="white" stroke-width="{max(1,block_r*0.35):.1f}" '
                        f'stroke-linecap="round"/>\n')

        else:
            svg += _action_glyph(step.instruction, cx, cy, glyph_size, color, comp)

        # Step number — only safe ASCII digits, no special chars
        off = r * 0.3
        svg += (f'<text x="{cx+off:.1f}" y="{cy-off:.1f}" text-anchor="start" '
                f'font-size="{label_size:.0f}" fill="{comp}" '
                f'font-family="sans-serif" font-weight="bold">{step.step+1}</text>\n')

    # Ghost piece at end
    if steps and flag:
        last = steps[-1]
        final_fine = _map_point(last.hex_idx, c2f)
        if 0 <= final_fine < N and final_fine != start_fine:
            end_center = grid.hexes[final_fine].center
            renderer = _PIECE_RENDERERS.get(piece.piece_type, 'pawnPiece')
            svg += getattr(flag, renderer)(MapCord(end_center.x, end_center.y),
                                           scale=piece_scale * 0.7,
                                           piece_id=f"plan_end_{piece.id[:8]}",
                                           fill=pat_fill,
                                           attrs={"opacity": "0.4"})

    return svg
```

The key changes: every glyph is now a `<circle>`, `<line>`, `<rect>`, `<polygon>`, or `<path>` — nothing that requires Unicode or font support. The step number label is safe ASCII digits only. The blocked marker uses crossed `<line>` elements instead of ✘. The rotation indicators use proper arc + arrowhead geometry, which also communicates direction much more clearly than ↺/↻.

I think the pause glyphs are too small. it is ok to take up 75% of the hex. The same is true with the piece size

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Two tweaks — `glyph_size` and `piece_scale` both need to grow to fill ~75% of the hex:

```python
    glyph_size = r * 0.75        # was max(6, r * 0.45) — now fills 75% of hex
    label_size = max(5, r * 0.3) # bumped slightly too so numbers stay readable
    dot_r      = max(2, r * 0.15)
    block_r    = max(4, r * 0.25)

    if piece_scale is None:
        piece_scale = (r * 0.75) / 22.5  # chess pieces are centered on 22.5px, fill 75% of hex
```

The chess SVGs are drawn around a `(22.5, 22.5)` center with a natural footprint of about 45px, so `scale = target_radius / 22.5` gives the right fit. At `r * 0.75 / 22.5` the piece fills about 75% of the hex diameter.

Also worth bumping the start ring marker to match:

```python
        svg += svg_ring(center.x, center.y, r * 0.8, stroke=color,
                        stroke_width=max(1.5, r * 0.07))
```

And the `svg_pause` bars should scale up proportionally too — they were using `size * 0.2` width and `size * 0.6` height, which at the new larger `glyph_size` already works correctly since they're relative. But you can also make them a bit chunkier:

```python
def svg_pause(cx, cy, size, fill, stroke, opacity=0.85):
    """Two vertical bars for PAUSE — sized to fill ~75% of hex."""
    w, h = size * 0.28, size * 0.75  # wider and taller than before
    for dx in [-size*0.22, size*0.22]:
        yield (f'<rect x="{cx+dx-w/2:.1f}" y="{cy-h/2:.1f}" '
               f'width="{w:.1f}" height="{h:.1f}" '
               f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
               f'rx="{w*0.25:.1f}" opacity="{opacity}"/>\n')
```

Same principle applies to `svg_shield`, `svg_sheaf`, etc. — they all receive `glyph_size` as their `size` argument so they'll automatically scale up with the one change to `glyph_size = r * 0.75`.

can you rewrite the method

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
def piece_plan_overlay(steps: list[PieceStep], grid: HexGrid,
                       piece: Piece = None, c2f: dict = None,
                       piece_scale: float = None) -> str:
    """Render movement plan — pure SVG geometry, no emoji or text glyphs."""
    svg, N = "", len(grid.hexes)
    flag = piece.flag if piece else None
    r = grid.radius

    # Fill ~75% of hex diameter
    if piece_scale is None:
        piece_scale = (r * 0.75) / 22.5  # chess SVGs are centered on 22.5px

    color = flag.primary if flag else "#4CAF50"
    comp  = flag.comp    if flag else "#333"

    # Heraldic fill pattern for chess pieces
    pat_fill = color
    if flag:
        pat_name = f"piece_pat_{piece.id[:8]}"
        pat = flag.flagPattern(pat_name, scale=0.1)
        grid.builder.add_definition(pat)
        pat_fill = f"url(#{pat_name})"

    glyph_size = r * 0.75            # fills 75% of hex
    label_size = max(5, r * 0.3)
    dot_r      = max(2, r * 0.15)
    block_r    = max(4, r * 0.25)

    arrow_id = f"plan_arrow_{id(piece) % 9999}"
    arrow_style = StyleCSS(arrow_id,
                           stroke=color, stroke_width=max(1, r * 0.08),
                           fill="none", opacity="0.7",
                           stroke_dasharray=f"{r*0.25:.0f},{r*0.12:.0f}")
    grid.builder.add_style(arrow_style)

    # Start piece + ring marker
    start_fine = _map_point(piece.location, c2f) if piece and piece.location is not None else -1
    if 0 <= start_fine < N and flag:
        center = grid.hexes[start_fine].center
        renderer = _PIECE_RENDERERS.get(piece.piece_type, 'pawnPiece')
        svg += getattr(flag, renderer)(MapCord(center.x, center.y),
                                       scale=piece_scale,
                                       piece_id=f"plan_start_{piece.id[:8]}",
                                       fill=pat_fill)
        svg += svg_ring(center.x, center.y, r * 0.8, stroke=color,
                        stroke_width=max(1.5, r * 0.07))

    for step in steps:
        fine_idx  = _map_point(step.hex_idx, c2f)
        fine_prev = _map_point(step.prev_idx, c2f) if step.prev_idx >= 0 else -1
        if fine_idx < 0 or fine_idx >= N:
            continue
        cx, cy = grid.hexes[fine_idx].center.x, grid.hexes[fine_idx].center.y

        # === FORWARD → spline arrow + waypoint dot ===
        if (step.instruction == Instruction.FORWARD and not step.blocked
                and 0 <= fine_prev < N and fine_prev != fine_idx):
            svg += grid.arrow(fine_prev, fine_idx, style=arrow_style, factor=0.3)
            svg += svg_circle(cx, cy, dot_r, fill=color, stroke="none", opacity=0.55)

        # === Blocked → red circle with white X ===
        elif step.blocked:
            svg += svg_circle(cx, cy, block_r, fill="red", stroke="darkred",
                              stroke_width=1, opacity=0.55)
            d = block_r * 0.55
            for x1,y1,x2,y2 in [(cx-d,cy-d,cx+d,cy+d),(cx+d,cy-d,cx-d,cy+d)]:
                svg += (f'<line x1="{x1:.1f}" y1="{y1:.1f}" '
                        f'x2="{x2:.1f}" y2="{y2:.1f}" '
                        f'stroke="white" stroke-width="{max(1, block_r*0.35):.1f}" '
                        f'stroke-linecap="round"/>\n')

        # === Action glyph — pure geometry ===
        else:
            svg += _action_glyph(step.instruction, cx, cy, glyph_size, color, comp)

        # Step number — safe ASCII only
        off = r * 0.35
        svg += (f'<text x="{cx+off:.1f}" y="{cy-off:.1f}" text-anchor="start" '
                f'font-size="{label_size:.0f}" fill="{comp}" '
                f'font-family="sans-serif" font-weight="bold">{step.step+1}</text>\n')

    # Ghost piece at final position
    if steps and flag:
        last = steps[-1]
        final_fine = _map_point(last.hex_idx, c2f)
        if 0 <= final_fine < N and final_fine != start_fine:
            end_center = grid.hexes[final_fine].center
            renderer = _PIECE_RENDERERS.get(piece.piece_type, 'pawnPiece')
            svg += getattr(flag, renderer)(MapCord(end_center.x, end_center.y),
                                           scale=piece_scale * 0.7,
                                           piece_id=f"plan_end_{piece.id[:8]}",
                                           fill=pat_fill,
                                           attrs={"opacity": "0.4"})
    return svg
```

In [ ]:
webMe(Div(NotStr(zoomed.hexGrid.builder.xml())))


In [ ]:
for s in steps:
    elev = terrain.elevations[s.hex_idx]
    ctry = terrain.fields["country"][s.hex_idx]
    tag = " ✘" if s.blocked else ""
    print(f"  {s.step+1:2d}. hex {s.hex_idx}  elev={elev:.1f}  country={ctry:.0f}{tag}")


## Put it together

In [ ]:
@rt("/settlement/{id}")
def settlement(session, id: str):
    """Full-page shell for settlement view — panels load via HTMX."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return RedirectResponse('/', status_code=303)

    return Titled("Settlement",
        Div(
            # Left sidebar — breadcrumbs + citizens
            Div(
                id="left-panel",
                hx_get=f"/settlement_left/{id}",
                hx_trigger="load",
                hx_indicator="#spinner",
                cls="w-64 bg-base-200 p-4 flex-shrink-0 overflow-y-auto"
            ),

            # Main area
            Div(
                Loading(cls=(LoadingT.spinner, LoadingT.lg),
                        htmx_indicator=True, id="spinner"),
                Div(id="map",
                    hx_get=f"/settlement_map/{id}",
                    hx_trigger="load",
                    hx_indicator="#spinner",
                    cls="flex-1 overflow-auto"),
                # ← wired up here
                Div(id="map-controls",
                    hx_get=f"/settlement_controls/{id}",
                    hx_trigger="load",
                    cls="h-16 bg-base-300 p-2 flex-shrink-0"),
                cls="flex-1 flex flex-col items-center justify-center relative"
            ),

            # Right sidebar — detail card
            Div(
                H4("Settlement", cls="text-sm font-bold mb-2"),
                Div(id="right-panel",
                    hx_get=f"/settlement_detail/{id}",
                    hx_trigger="load",
                    hx_indicator="#spinner"),
                cls="w-64 bg-base-200 p-4 flex-shrink-0 overflow-y-auto"
            ),
            cls="flex h-screen"
        )
    )


## Route Picker

I am going to work through some thoughts

## Modified Gosper
so the rules on Piece is going to be reworked as a list of ints with the following
0 -> pause
1 -> defend # in defend mode we get more aggressive
2 -> harvest # gather food from the hex
3 -> give   # give food to either a neighbor piece or a settlement
4 -> settle or siege # reserved for figuring out how to conquer settlements
5 -> forward # move on place forward (depending on movement cost). If you can't move forward (ie you hit water) rotate left. if it is there is another piece in the way pause   
6 -> rotate left
7 -> rotate right


## Movement costs

we have a movement cost formula
```python
@patch
def movement_cost(self: GameBoard, from_pos: HexPosition, to_pos: HexPosition, origin: int) -> float:
    """Calculate cost to move from one hex to another using HexPositions.
    
    Args:
        from_pos: Starting HexPosition
        to_pos: Destination HexPosition
        origin: Origin index for converting HexPosition to grid index
    
    Returns infinity for invalid moves (water, out of bounds).
    """
    terrain = self.terrain
    grid = terrain.hexGrid
    
    # Convert to indices to check terrain
    to_idx = grid.hexposition_to_index(to_pos, origin)
    from_idx = grid.hexposition_to_index(from_pos, origin)
    
    # Check bounds
    if to_idx < 0 or to_idx >= len(terrain.elevations):
        return float('inf')
    
    # Can't cross water
    if terrain.elevations[to_idx] < 1:
        return float('inf')
    
    from_elev = terrain.elevations[from_idx]
    to_elev = terrain.elevations[to_idx]
    
    # Base cost
    base_cost = 1.0
    
    # Elevation gain penalty (climbing is expensive)
    elev_diff = to_elev - from_elev
    if elev_diff > 0:
        # Exponential penalty for climbing
        base_cost += elev_diff * 2.0
    
    return base_cost
```
Which we will need to figure out how to use for a turn. movement_range will tell how many moves we can do. Lets make turns free (including turns from hitting the ocean). the first 4 counts as a single movement cost.

## food and diet
we are going to add three fields 
1. which is the amount of food a piece has
2. its diet. how much does it eat per turn If it rounds out of food it will start to lose health. 
3. food capacity which is how much total food it can carry.

## deprecate
we don't need memory since that will be in rules.

## settle
We are going to keep these around even though I haven't figure them out
harvest_goal: Resources = None
settle_progress: int = 0
settle_threshold: int = 3
sight = 3

## The things we carry
Lets start thinking about the rules editor for a piece
1. we are going to want a list of rules that are assigned and be able to add and delete rows
2. we are going to want an edit point where we can insert new items
3. we are going to want a HexDragMap that we can add items. The path we make needs to be converted into instructions on how we get there.
4. we need an apply button and the number of rounds. This will be our main map showing where the piece is going to move each turn for the number of turns. I envision this as a series of arrows and circles.
a. arrows are for movement. straight line if single cost squiggle line if multiple cost. the color of the arrow is how aggressive it is.
b. circles are for the pause actions (0 -> pause
1 -> defend # in defend mode we get more aggressive
2 -> harvest # gather food from the hex
3 -> give   # give food to either a neighbor piece or a settlement
) I think red should be for defend.

eventually we are going to want a drop down of a list of common rule sets (patrol around a circle. go out and harvest and comeback, attack by first flanking left). 
we will certainly want to have a hex destination that returns a set of rules on how to get there (hello djisktra)

I need to be clearer on some definitons
1. `round` is for all player and all pieces. Each player gives the set of orders it wants its pieces to do. We need to build out how to carry them out, but I am tempted just to start with the kindergaren model of everybody does what it wants
2. `turn` is equivalent to a round, but just on a single piece. `movement_cost` is how many things it can do in a turn. Some (like rotation) are free. Others like mmoving up a hill might cost a couple of movements.

It can be the case that rule ends and there are spare items. So if the last item in the list is one of the non movement items. ie (like pause) then we just pause the remaining amount of time. if it is a movement (or rotate) then we circle back to the begining.

## Pulling Ranks
so things are prioritzed in chess order ie kings move first and eat first. then queens. This way if a king will move to a spot before a pawn would

I am going to have the flow of the hex be the amount you can harvest. Settlments are going to have build queues. I need to figure out the mechanics of adding settlements whether there are pre existing locations or it behavior on a item

**7. Food economy**

Ill need a `hex_food_yield(terrain, idx) -> float` keyed off climate and flow.



**3. Forward-blocked spin trap**

Instruction 4 says "rotate left if water, pause if piece." But imagine a piece in a cove with water on 5 sides — it rotates 5 times but each time if the chance to move becomes available



## Piece Editor — Progress & Roadmap

### What we built today

**1. Facing Selector (`facing_grid`)**
- 1-ring `HexGrid.centered` compass rose with `HexTouchMap`
- Center hex shows the piece, 6 neighbors show direction arrows (↙←↖↗→↘)
- Tapping a direction hex POSTs to `/set_facing/{piece_id}/{facing}` and swaps the widget
- Current facing highlighted in green (`fc-active` style)
- Direction indices match `HexPosition.directions()`: [SW=0, W=1, NW=2, NE=3, E=4, SE=5]

**2. Drag-to-Rules Editor (`piece_drag_editor`)**
- 3-ring `HexGrid.centered` with `HexDragMap` for drag selection
- Amber piece hex at center with facing arrow overlay
- Drag across hexes → JS collects `hex_ids` → POSTs to `/piece_path/{piece_id}/{facing}`
- Server reconstructs same grid, converts indices to `path_to_rules()`

**3. `path_to_rules()` converter**
- Takes list of hex grid indices + starting facing
- Deduplicates consecutive repeats from drag
- For each step: computes delta HexPosition, finds target direction index
- Generates shortest rotation (ROT_L/ROT_R) then FORWARD
- Verified: center→E→NE→NE with facing=E produces `[5, 6, 6, 5, 7, 5]` ✓

**4. `rules_display()` — badge visualization**
- Colour-coded instruction badges (blue=forward, grey=rotate, red=defend, green=harvest, etc.)
- Summary line: "6 instructions — 3 moves, 3 turns"
- Raw rule string shown in mono for debugging

**5. Routes**
- `GET /piece_editor_demo` — standalone test page with drag editor + facing selector
- `POST /piece_path/{piece_id}/{facing}` — receives drag, returns rules badges
- `GET /facing_selector/{piece_id}` — renders compass for a DB piece
- `POST /set_facing/{piece_id}/{facing}` — updates facing, returns refreshed compass
- Demo pieces (id starts with "demo-") skip DB persistence

**6. Instruction enum (in data.py)**


In [ ]:
#| export
# ── 1. Path → rules converter ─────────────────────────────────

def path_to_rules(hex_indices: list[int], grid: HexGrid, start_facing: int = 0) -> list[int]:
    """Convert a list of hex grid indices (from drag) into Instruction ints."""
    if len(hex_indices) < 2:
        return []

    dirs = HexPosition.directions()   # [SW, W, NW, NE, E, SE] = indices 0–5

    # Deduplicate consecutive duplicates (drag can repeat hexes)
    clean = [hex_indices[0]]
    for idx in hex_indices[1:]:
        if idx != clean[-1]:
            clean.append(idx)

    rules = []
    facing = start_facing

    for i in range(len(clean) - 1):
        from_pos = grid.index_to_hexposition(clean[i],   clean[0])
        to_pos   = grid.index_to_hexposition(clean[i+1], clean[0])
        delta    = to_pos - from_pos

        # Find which direction matches the delta
        try:
            target_dir = next(j for j, d in enumerate(dirs) if d == delta)
        except StopIteration:
            # Non-adjacent hex in drag — skip
            continue

        # Shortest rotation from facing to target_dir
        diff = (target_dir - facing) % 6
        if diff == 0:
            pass                                        # already facing right
        elif diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))

        rules.append(Instruction.FORWARD.value)
        facing = target_dir

    return rules

In [ ]:
#| export
# ── 2. Rules → readable labels ────────────────────────────────

_INSTR_LABELS = {
    0: ("⏸ Pause",   "#94a3b8"),
    1: ("🛡 Defend",  "#ef4444"),
    2: ("🌾 Harvest", "#22c55e"),
    3: ("🎁 Give",    "#a855f7"),
    4: ("🏰 Settle",  "#f59e0b"),
    5: ("→ Forward", "#3b82f6"),
    6: ("↺ Rot L",   "#64748b"),
    7: ("↻ Rot R",   "#64748b"),
}

def rules_display(rules: list[int], piece_id: str):
    """Compact badge list for a rule sequence."""
    if not rules:
        return P("No rules yet — drag a path above", cls="text-sm opacity-50")

    badges = []
    for i, r in enumerate(rules):
        label, color = _INSTR_LABELS.get(r, (str(r), "#888"))
        badges.append(
            Span(label,
                 style=f"background:{color}22; border:1px solid {color}; "
                        "border-radius:4px; padding:2px 6px; font-size:11px; "
                       f"color:{color}; margin:2px; display:inline-block;")
        )

    return Div(
        P(f"{len(rules)} instructions", cls="text-xs opacity-50 mb-1"),
        Div(*badges, style="line-height:2"),
        id=f"rules-{piece_id[:8]}",
    )

In [ ]:
#| export
# ── 3. The drag map widget ─────────────────────────────────────

def piece_drag_editor(piece: Piece, rings: int = 3, radius: int = 35):
    """
    HexDragMap centered on piece location.
    Drag a path → POST hex_ids → convert to rules → display.
    """
    base    = StyleCSS("pe-hex",    fill="#1e293b", stroke="#475569", stroke_width=1)
    current = StyleCSS("pe-cur",    fill="#f59e0b", stroke="#fbbf24", stroke_width=2)
    lbl     = StyleCSS("pe-label",  fill="#e2e8f0", font_size="11px")

    grid = HexGrid.centered(rings, radius=radius, style=base)
    for s in [current, lbl]:
        grid.builder.add_style(s)

    mid = grid.middle

    # Mark piece position
    grid.hexes[mid].style = current
    grid.hexes[mid].label = piece.piece_type.value[0].upper()
    grid.hexes[mid].labelStyle = lbl.name

    # Draw facing arrow from center
    facing_dir = HexPosition.directions()[piece.facing % 6]
    facing_idx = grid.hexposition_to_index(facing_dir, mid)
    if facing_idx >= 0:
        arrow_layer = grid.arrow(mid, facing_idx,
                                 style=StyleCSS("pe-arrow", stroke="#f59e0b",
                                                stroke_width=2))
        grid.builder.adjust("facing", arrow_layer)

    grid.update()

    widget_id = f"drag-editor-{piece.id[:8]}"

    return Div(
        H5("Drag a patrol path", cls="text-sm font-bold mb-1"),
        P("Drag across hexes to generate movement instructions",
          cls="text-xs opacity-50 mb-2"),
        
        ##
        HexDragMap(
            grid,
            on_drag={
                "hx_post":   f"/piece_path/{piece.id}/{piece.facing}",
                "hx_target": f"#rules-{piece.id[:8]}",
                "hx_swap":   "outerHTML",
            },
            id=widget_id,
            # removed style="touch-action:none;" — HexDragMap adds it already
        ),

        ##
        rules_display(piece.rules, piece.id),
        cls="space-y-2"
    )

In [ ]:
#| export
# ── 4. Route that receives the drag ───────────────────────────

@rt("/piece_path/{piece_id}/{facing:int}")
async def piece_path(request, piece_id: str, facing: int):
    """Receive dragged hex_ids, convert to rules, persist, return badge display."""
    form = await request.form()
    raw  = form.get("hex_ids", "")

    if not raw:
        return Div(P("No path received", cls="text-sm opacity-50"),
                   id=f"rules-{piece_id[:8]}")

    hex_ids = [int(x) for x in raw.split(",") if x.strip()]

    # We need the grid to convert indices → HexPositions.
    # Use a fresh centered grid matching piece_drag_editor's rings/radius.
    rings  = 3
    radius = 35
    dummy_style = StyleCSS("x", fill="white")
    grid = HexGrid.centered(rings, radius=radius, style=dummy_style)

    new_rules = path_to_rules(hex_ids, grid, start_facing=facing)

    # Persist to DB if we can find the piece
    try:
        rec = globalStore.pieces[piece_id]
        if rec:
            existing = [int(x) for x in (rec.get("rules") or "").split(",") if x]
            merged = existing + new_rules   # append; swap for replace if preferred
            globalStore.pieces.upsert(
                {"id": piece_id, "rules": ",".join(str(r) for r in merged)},
                pk="id"
            )
            new_rules = merged
    except Exception:
        pass   # demo mode — just show without saving

    dummy = Piece(id=piece_id, rules=new_rules)
    return rules_display(dummy.rules, piece_id)

In [ ]:
#| export
dummy_rook = Piece(
    id   = "demo-rook-0000-0000-000000000000",
    piece_type = PieceType.ROOK,
    facing     = 4,   # starts facing E
    rules      = [],
)

webMe(Div(
    H3("Rook Rule Editor"),
    piece_drag_editor(dummy_rook),
    cls="p-4 space-y-4 bg-slate-900 text-white"
))

the drag-to-path converter. We could pratice with a dummy piece maybe a rook.

In [ ]:
def showActiveSettlements(session):
    uid = ensure_user(session)
    logging.info(f"showSettlement: uid={uid}")
    active = globalStore.active_board(uid)
    ret = []
    for country in active.board.kingdoms:
        ret.extend( country.settlements)
    return ret

In [ ]:
showActiveSettlements(dummySession)

In [ ]:
# Simulate a drag path: center → E → NE → NE
grid = HexGrid.centered(3, radius=35, style=StyleCSS("x", fill="white"))
mid = grid.middle  # 24

# Manually pick a path: center, then one E, then one NE
e_idx  = grid.hexposition_to_index(HexPosition.E, mid)
ne_idx = grid.hexposition_to_index(HexPosition.NE, mid)
ne2    = grid.hexposition_to_index(HexPosition.NE + HexPosition.NE, mid)

drag_path = [mid, e_idx, ne_idx, ne2]
print(f"Drag indices: {drag_path}")

rules = path_to_rules(drag_path, grid, start_facing=4)  # facing E
labels = [_INSTR_LABELS[r][0] for r in rules]
print(f"Rules: {rules}")
print(f"Labels: {labels}")


Is it working?

Yes  wire up the /piece_editor_demo

In [ ]:
#| export
def rules_display(rules: list[int], piece_id: str):
    """Compact badge list for a rule sequence, always with correct swap id."""
    target_id = f"rules-{piece_id[:8]}"

    if not rules:
        return Div(
            P("No rules yet — drag a path above", cls="text-sm opacity-50"),
            id=target_id,
        )

    badges = []
    for i, r in enumerate(rules):
        label, color = _INSTR_LABELS.get(r, (str(r), "#888"))
        badges.append(
            Span(label,
                 style=f"background:{color}22; border:1px solid {color}; "
                        "border-radius:4px; padding:2px 6px; font-size:11px; "
                       f"color:{color}; margin:2px; display:inline-block;")
        )

    # Count summary
    forwards = sum(1 for r in rules if r == 5)
    rotations = sum(1 for r in rules if r in (6, 7))
    actions = sum(1 for r in rules if r in (0, 1, 2, 3, 4))

    summary = f"{forwards} moves, {rotations} turns"
    if actions:
        summary += f", {actions} actions"

    return Div(
        P(f"{len(rules)} instructions — {summary}", cls="text-xs opacity-50 mb-1"),
        Div(*badges, style="line-height:2"),
        Pre(",".join(str(r) for r in rules),
            cls="text-xs opacity-40 mt-2 font-mono"),
        id=target_id,
    )


In [ ]:
#| export
@rt("/piece_editor_demo")
def piece_editor_demo(session):
    """Live demo page for the drag-to-rules editor."""
    try:
        logging.info("piece_editor_demo: building page")
        editor = piece_drag_editor(dummy_rook)
        logging.info("piece_editor_demo: editor built OK")
        facing = facing_grid(dummy_rook.id, current_facing=dummy_rook.facing)
        logging.info("piece_editor_demo: facing built OK")
        return Titled("Piece Editor Demo",
            Div(
                H3("Rook Rule Editor"),
                P("Drag across hexes to create movement rules.",
                  cls="text-sm opacity-70 mb-4"),
                editor,
                Divider(),
                H4("Facing Selector"),
                facing,
                cls="p-6 space-y-6 max-w-xl mx-auto"
            )
        )
    except Exception as e:
        logging.error(f"piece_editor_demo FAILED: {e}", exc_info=True)
        return P(f"Error: {e}")


@rt("/piece_path/{piece_id}/{facing:int}")
async def piece_path(request, piece_id: str, facing: int):
    """Receive dragged hex_ids, convert to rules, return badge display."""
    try:
        form = await request.form()
        raw = form.get("hex_ids", "")
        logging.info(f"piece_path: piece_id={piece_id} facing={facing} raw='{raw}'")

        if not raw:
            return Div(P("No path received", cls="text-sm opacity-50"),
                       id=f"rules-{piece_id[:8]}")

        hex_ids = [int(x) for x in raw.split(",") if x.strip()]
        logging.info(f"piece_path: {len(hex_ids)} hex_ids: {hex_ids}")

        # Reconstruct same grid as piece_drag_editor
        rings  = 3
        radius = 35
        dummy_style = StyleCSS("x", fill="white")
        grid = HexGrid.centered(rings, radius=radius, style=dummy_style)

        new_rules = path_to_rules(hex_ids, grid, start_facing=facing)
        logging.info(f"piece_path: {len(new_rules)} rules: {new_rules}")

        # For demo pieces, skip DB persistence
        if not piece_id.startswith("demo-"):
            try:
                rec = globalStore.pieces[piece_id]
                if rec:
                    existing_str = rec.rules if hasattr(rec, 'rules') else rec.get("rules", "")
                    existing = [int(x) for x in (existing_str or "").split(",") if x]
                    new_rules = existing + new_rules
                    globalStore.pieces.upsert(
                        {"id": piece_id, "rules": ",".join(str(r) for r in new_rules)},
                        pk="id"
                    )
            except Exception as db_err:
                logging.warning(f"piece_path: DB skip: {db_err}")

        return rules_display(new_rules, piece_id)

    except Exception as e:
        logging.error(f"piece_path FAILED: {e}", exc_info=True)
        return Div(P(f"Error: {e}", cls="text-sm text-red-500"),
                   id=f"rules-{piece_id[:8]}")


In [ ]:
!tail -10 base.text

This works and I am so excited. I want to do a commit which means I will loose the prompt history. can you write up where we have been and where we are going so I can put it into a note to come back to.

## Piece Editor — Progress & Roadmap

### What we built today

**1. Facing Selector (`facing_grid`)**
- 1-ring `HexGrid.centered` compass rose with `HexTouchMap`
- Center hex shows the piece, 6 neighbors show direction arrows (↙←↖↗→↘)
- Tapping a direction hex POSTs to `/set_facing/{piece_id}/{facing}` and swaps the widget
- Current facing highlighted in green (`fc-active` style)
- Direction indices match `HexPosition.directions()`: [SW=0, W=1, NW=2, NE=3, E=4, SE=5]

**2. Drag-to-Rules Editor (`piece_drag_editor`)**
- 3-ring `HexGrid.centered` with `HexDragMap` for drag selection
- Amber piece hex at center with facing arrow overlay
- Drag across hexes → JS collects `hex_ids` → POSTs to `/piece_path/{piece_id}/{facing}`
- Server reconstructs same grid, converts indices to `path_to_rules()`

**3. `path_to_rules()` converter**
- Takes list of hex grid indices + starting facing
- Deduplicates consecutive repeats from drag
- For each step: computes delta HexPosition, finds target direction index
- Generates shortest rotation (ROT_L/ROT_R) then FORWARD
- Verified: center→E→NE→NE with facing=E produces `[5, 6, 6, 5, 7, 5]` ✓

**4. `rules_display()` — badge visualization**
- Colour-coded instruction badges (blue=forward, grey=rotate, red=defend, green=harvest, etc.)
- Summary line: "6 instructions — 3 moves, 3 turns"
- Raw rule string shown in mono for debugging

**5. Routes**
- `GET /piece_editor_demo` — standalone test page with drag editor + facing selector
- `POST /piece_path/{piece_id}/{facing}` — receives drag, returns rules badges
- `GET /facing_selector/{piece_id}` — renders compass for a DB piece
- `POST /set_facing/{piece_id}/{facing}` — updates facing, returns refreshed compass
- Demo pieces (id starts with "demo-") skip DB persistence

**6. Instruction enum (in data.py)**


can you add a rook piece to 5642d99d-fbbe-4746-81ff-7e8f8552e3ca

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Sure! Here's the code to create a rook and attach it to that settlement:

```python
settle_id = "5642d99d-fbbe-4746-81ff-7e8f8552e3ca"
place = globalStore.settlement_from_id(settle_id)

rook = Piece(
    piece_type=PieceType.ROOK,
    owner_id=place.owner_id,
    location=place.location,
    facing=4,  # E
)

place.citizens.append(rook)
rook.save(db=globalStore, world_id=1, settlement_id=settle_id, kingdom_id=place.owner_id)

print(f"Created rook {rook.id[:12]}… at location {rook.location}, kingdom {rook.owner_id}")
print(f"Settlement now has {len(place.citizens)} citizen(s)")
```

This creates a ROOK with default stats (attack=7, move=2, harvest=1, scout=1), places it at the settlement's hex, and persists it to the DB. The `world_id=1` matches what I see in your settlement records.

settle_id = "5642d99d-fbbe-4746-81ff-7e8f8552e3ca"
place = globalStore.settlement_from_id(settle_id)

rook = Piece(
    piece_type=PieceType.KNIGHT,
    owner_id=place.owner_id,
    location=place.location,
    facing=4,  # E
)

place.citizens.append(rook)
rook.save(db=globalStore, world_id=1, settlement_id=settle_id, kingdom_id=place.owner_id)

print(f"Created rook {rook.id[:12]}… at location {rook.location}, kingdom {rook.owner_id}")
print(f"Settlement now has {len(place.citizens)} citizen(s)")


can you tell me all of the pieces of  "5642d99d-fbbe-4746-81ff-7e8f8552e3ca"

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
cursor = globalStore.db.execute(
    "SELECT id, piece_type, location, health, food, facing FROM piece_record WHERE settlement_id = ?",
    [settle_id]
)
cols = [d[0] for d in cursor.description]
pd.DataFrame([dict(zip(cols, r)) for r in cursor.fetchall()])
```

In [ ]:
homeSettle = [x for x in showActiveSettlements(dummySession) if x.id == "5642d99d-fbbe-4746-81ff-7e8f8552e3ca"][0]

In [ ]:
homeSettle

In [ ]:
knight =  homeSettle.citizens[0]

In [ ]:
knight

We need to adjust knight so that it has its parents_id  (from the '5642d99d-fbbe-4746-81ff-7e8f8552e3ca' and we should save that to the database as well

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



```python
knight.parent_id = homeSettle.id
knight.save(db=globalStore, world_id=1, settlement_id=homeSettle.id, kingdom_id=homeSettle.owner_id)
print(f"Knight parent_id set to {knight.parent_id[:12]}…")
```

One thing to note — looking at the `PieceRecord` dataclass and the `save` patch, there's no `parent_id` column in `PieceRecord`. The field exists on `Piece` but doesn't get persisted. Want me to sketch out adding it to `PieceRecord` so it round-trips through the DB?

In [ ]:
knight.parent_id = homeSettle.id
knight.save(db=globalStore, world_id=1, kingdom_id=homeSettle.owner_id)
print(f"Knight parent_id set to {knight.parent_id[:12]}…")


In [ ]:
knight

I think some of the conversation about the simulator wasn't clear when I was doing the prompts. can you write up the approach I am aiming for so I can put it at the top. 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Whoops! An error (prompt is too long: 263420 tokens > 200000 maximum) occurred while processing your request.
If this problem persists, please contact us on Discord.
Please include your dialog url and error info in your message.